# 0d Heatbath

Set up a heatbath with only one cell and initialize the cell firstly with only one species, but with a strong thermal non-equilibrium. This follows the verification strategy of Casseau, V. 2021.

## Heating (T > T_V)

In [1]:
%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp

import compressible.chemistry_types as chemistry_types
import compressible.chemistry_utils as chemistry_utils
import compressible.constants as constants
import compressible.energy_models as energy_models
from compressible.boundary_conditions_utils import build_boundary_arrays_1d_periodic
from compressible.equation_manager import run_scan
from compressible.equation_manager_types import EquationManager
from compressible.mesh import Mesh
from compressible.numerics_types import ClippingConfig, NumericsConfig
from compressible.state import compute_U_from_primitives, extract_primitives_from_U


T_tr_init = 10000.0  # K
T_V_init = 1000.0  # K

p_init = 1.0 * constants.ATM_TO_PA

general_species_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_gnoffo.json"
)
gnoffo_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/"
    "air_5_gnoffo_equilibrium_enthalpy.json"
)
bird_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_bird_energy.json"
)
species_names = ["N2"]

dt = 1e-9  # s
t_final = 1e-5  # s
save_interval = 1  # every x steps
dx = 1e-4  # m

# ------------------------------ Simulation setup ------------------------------

energy_model_config_gnoffo = energy_models.EnergyModelConfig(
    model="gnoffo",
    include_electronic=True,
    data_path=gnoffo_equilibrium_enthalpy_data_path,
)

energy_model_config_bird = energy_models.EnergyModelConfig(
    model="bird",
    include_electronic=False,
    data_path=bird_equilibrium_enthalpy_data_path,
)

species = chemistry_utils.load_species_table(
    general_data_path=general_species_data_path,
    species_names=species_names,
    energy_model_config=energy_model_config_gnoffo,
)

mesh = Mesh.from_1d_grid(jnp.array([0.0, dx]), periodic=True)
boundary_arrays = build_boundary_arrays_1d_periodic(mesh, species.n_species)

numerics_config = NumericsConfig(
    dt=dt,
    cfl=0.4,
    dt_mode="fixed",
    integrator_scheme="forward-euler",
    spatial_scheme="first_order",
    flux_scheme="hllc",
    clipping=ClippingConfig(),
)

equation_manager = EquationManager(
    species=species,
    reactions=None,
    numerics_config=numerics_config,
    boundary_arrays=boundary_arrays,
)

rho_init = p_init * species.molar_masses[0] / (constants.R_universal * T_tr_init)

U_init = compute_U_from_primitives(
    Y_s=jnp.array([[1.0]]),
    rho=jnp.array([rho_init]),
    u=jnp.array([0.0]),
    v=jnp.zeros(1),
    T_tr=jnp.array([T_tr_init]),
    T_V=jnp.array([T_V_init]),
    equation_manager=equation_manager,
)

U_field, t = run_scan(
    U_init=U_init,
    mesh=mesh,
    equation_manager=equation_manager,
    t_final=t_final,
    save_interval=save_interval,
)
print("Simulation completed.")

Simulation completed.


In [11]:
from plotly import graph_objects as go
import pandas as pd

# JIT-compiled version of extract_primitives_from_U (needs to be executed once per kernel uptime)
def _extract_prim(U, em):
    prim = extract_primitives_from_U(U, em)
    return prim.Y_s, prim.rho, prim.T, prim.Tv, prim.p

extract_primitives_from_U_jitted = jax.jit(_extract_prim)

# extract primitives and plot
Y_s, rho, T, T_V, p = jax.vmap(
    extract_primitives_from_U_jitted,
    in_axes=(0, None),
)(U_field, equation_manager)


fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=t,
        y=T_V[:, 0],
        mode="lines",
        name="T_V",
        line=dict(shape="spline", smoothing=1.0, width=4),
    )
)
fig.add_trace(
    go.Scatter(
        x=t, y=T[:, 0], mode="lines", name="T", line=dict(shape="spline", smoothing=1.0, width=4)
    )
)

fig.add_hline(y=7623.3, name="T_eq Casseau", line_dash="dash", line_color="black")

# Load and plot reference data from CSV
csv_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_1.csv"
df = pd.read_csv(
    csv_path, skiprows=1
)  # Skip the first header row, use X,Y,X,Y... as columns

# Read first row to get the dataset names
with open(csv_path, "r") as f:
    header_line = f.readline().strip()
dataset_names = [name for name in header_line.split(",") if name]

# Group datasets by prefix (e.g., "modified", "hy2foam_default", "monaco")
prefixes = []
for name in dataset_names:
    # Extract prefix by removing _t_v or _t_tr suffix
    if "_t_v" in name:
        prefix = name.replace("_t_v", "")
    elif "_t_tr" in name:
        prefix = name.replace("_t_tr", "")
    else:
        prefix = name
    if prefix not in prefixes:
        prefixes.append(prefix)

# Define colors for each prefix
colors = ["green", "red", "purple", "orange", "brown", "pink"]
prefix_colors = {prefix: colors[i % len(colors)] for i, prefix in enumerate(prefixes)}

# Plot each dataset (columns come in pairs: X, Y for each dataset)
for i, name in enumerate(dataset_names):
    x_col = i * 2  # X column index
    y_col = i * 2 + 1  # Y column index

    if x_col >= len(df.columns) or y_col >= len(df.columns):
        continue

    x_data = pd.to_numeric(df.iloc[:, x_col], errors="coerce").dropna().values
    y_data = pd.to_numeric(df.iloc[:, y_col], errors="coerce").dropna().values

    # Use minimum length in case of mismatched data
    min_len = min(len(x_data), len(y_data))
    x_data = x_data[:min_len]
    y_data = y_data[:min_len]

    # Find prefix for this dataset
    if "_t_v" in name:
        prefix = name.replace("_t_v", "")
    elif "_t_tr" in name:
        prefix = name.replace("_t_tr", "")
    else:
        prefix = name

    fig.add_trace(
        go.Scatter(
            x=x_data,
            y=y_data * 1000,  # Scale Y values (they appear to be in units of 1000 K)
            mode="markers+lines",
            line=dict(dash="dot", smoothing=1.0, shape="spline"),
            name=name,
            marker=dict(color=prefix_colors.get(prefix, "gray"), symbol="circle" if "_t_v" in name else "square" if "_t_tr" in name else "circle", size=12),
        )
    )

fig.update_xaxes(type="log", exponentformat="power", showexponent="all", showgrid=True)
fig.update_yaxes(range=[0, 10500], showgrid=True)
fig.update_layout(
    template="simple_white",
    # title="Energy Relaxation Correlation with Casseau for nonreacting N2",
    xaxis_title="Time (s)",
    yaxis_title="Temperature (K)",
    legend=dict(
        x=0.95,
        y=0.05,
        xanchor="right",
        yanchor="bottom",
        borderwidth=1,
        bordercolor="black",
    ),
    showlegend=True,
    width=800,
    height=800,
)
fig.show()

# fig.update_xaxes(range=[5e-7, 1e-6], type="linear", tickformat=".1e")
# fig.update_yaxes(range=[5700, 6300])
fig.update_layout(
    width=800,
    height=800,
    margin=dict(t=10, b=280, l=80, r=10),
    xaxis=dict(
        title_font=dict(size=26),
        tickfont=dict(size=22),
        tickangle=45,
    ),
    yaxis=dict(title_font=dict(size=26), tickfont=dict(size=22)),
    legend=dict(
        font=dict(size=22),
        x=0.5,
        y=-0.25,
        xanchor="center",
        yanchor="top",
        borderwidth=1,
        bordercolor="white",
        orientation="h",
    ),
)
fig.show()
fig.write_image(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/13.pdf"
)

### Conclusion
Good agreement with the results published by Casseau. The non-constant difference of T_V to the default hy2foam configuration potentially stems from the fact that hy2foam keeps the pressure constant at 1atm while in my setup pressure varies between 1.2 and 0.9 atm while the density is kept constant. The discontinuity of T_V at 7x10^-7 originates from a discontinuity in the enthalpy fits provided by Gnoffo. 

In [31]:
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from pathlib import Path

import compressible.energy_models as energy_models
import compressible.thermodynamic_relations as thermodynamic_relations
from compressible.chemistry_utils import load_species_table
from compressible.energy_models_utils import _load_gnoffo_energy_data

jax.config.update("jax_enable_x64", True)

import sys

species_name = "N2"  # enthalpy of this species will be plotted

# Add src directory to path
repo_root = Path.cwd().parent  # Assuming notebook is in experiments/
src_path = repo_root / "src"
sys.path.insert(0, str(src_path))

# Load data
general_species_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_gnoffo.json"
)

gnoffo_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/"
    "air_5_gnoffo_equilibrium_enthalpy.json"
)

energy_model_config_gnoffo = energy_models.EnergyModelConfig(
    model="gnoffo",
    include_electronic=True,
    data_path=gnoffo_equilibrium_enthalpy_data_path,
)

species_table = load_species_table(
    species_names=["N2"],
    general_data_path=general_species_data_path,
    energy_model_config=energy_model_config_gnoffo,
)
# Get species index
species_index = species_table.names.index(species_name)

T_limit_low, T_limit_high, _ = _load_gnoffo_energy_data(
    gnoffo_equilibrium_enthalpy_data_path, [species_name]
)

T = jnp.linspace(
    T_limit_low[0, 0],
    T_limit_high[0, -1],
    35000,
    endpoint=False,
)

# Compute enthalpy
h = thermodynamic_relations.compute_e_ve(
    T_V=T,
    species_table=species_table,
)

h_species = h[species_index, :]

# Convert to MJ/kg for better readability
h_species_MJ = h_species / 1e6

# Plot with Plotly
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=T,
        y=h_species_MJ,
        mode="lines",
        name=f"{species_name} (Enthalpy)",
        line=dict(color="blue", width=2),
    )
)

# Mark the temperature ranges
for T_range in T_limit_low[0, 1:]:  # Skip first one
    fig.add_vline(x=T_range, line_dash="dash", line_color="red", opacity=0.3)

fig.update_xaxes(range=[5900, 6100])
fig.update_yaxes(range=[1.35, 1.5])
fig.update_layout(
    title=f"Enthalpy vs Temperature for {species_name}",
    xaxis_title="Temperature [K]",
    yaxis_title="Enthalpy [MJ/kg]",
    hovermode="x unified",
    template="plotly_white",
    width=800,
    height=600,
)

fig.show()
fig.update_layout(
    title=None,
    width=800,
    height=800,
    margin=dict(t=10, b=100, l=80, r=10),
    xaxis=dict(
        title_font=dict(size=26),
        tickfont=dict(size=22),
        # tickangle=45,
    ),
    yaxis=dict(title_font=dict(size=26), tickfont=dict(size=22)),
    legend=dict(
        font=dict(size=22),
        x=0.5,
        y=-0.25,
        xanchor="center",
        yanchor="top",
        borderwidth=1,
        bordercolor="white",
        orientation="h",
    ),
)
fig.write_image(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/gnoffoenthalpyjump.pdf"
)

In [ ]:
# compute the degrees of freedom to plausibilize the equilibrium temperature
import compressible.constants as constants
import compressible.thermodynamic_relations as thermodynamic_relations

T_V_check = jnp.array([T_V_init, T_V[-1, 0]])
e_vib = thermodynamic_relations.compute_e_vib_electronic(
    T_V=T_V_check,
    T_ref=298.0,
    T_limit_low=species.T_limit_low,
    T_limit_high=species.T_limit_high,
    parameters=species.enthalpy_coeffs,
    is_monoatomic=species.is_monoatomic,
    molar_masses=species.molar_masses,
)
dof_vib = (
    2 * e_vib * species.molar_masses[0] * 1e-3 / (constants.R_universal * T_V_check)
)
print("Degrees of freedom vibrational at init and final:", dof_vib)

Degrees of freedom vibrational at init and final: [[0.2087732 1.6868243]]


## Cooling (T < T_V)

In [14]:

%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp

import compressible.chemistry as chemistry
import compressible.chemistry_types as chemistry_types
import compressible.chemistry_utils as chemistry_utils
import compressible.constants as constants
import compressible.energy_models as energy_models
from compressible.boundary_conditions_utils import build_boundary_arrays_1d_periodic
from compressible.equation_manager import run_scan
from compressible.equation_manager_types import EquationManager
from compressible.mesh import Mesh
from compressible.numerics_types import ClippingConfig, NumericsConfig
from compressible.state import compute_U_from_primitives, extract_primitives_from_U

T_tr_init = 3000.0  # K
T_V_init = 10000.0  # K

p_init = 1.0 * constants.ATM_TO_PA

general_species_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_gnoffo.json"
)
gnoffo_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/"
    "air_5_gnoffo_equilibrium_enthalpy.json"
)
bird_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_bird_energy.json"
)
species_names = ["N2"]

dt = 1e-8  # s
t_final = 1e-4  # s
save_interval = 1  # every x steps
dx = 1e-4  # m

# ------------------------------ Simulation setup ------------------------------

energy_model_config_gnoffo = energy_models.EnergyModelConfig(
    model="gnoffo",
    include_electronic=True,
    data_path=gnoffo_equilibrium_enthalpy_data_path,
)

energy_model_config_bird = energy_models.EnergyModelConfig(
    model="bird",
    include_electronic=False,
    data_path=bird_equilibrium_enthalpy_data_path,
)

species = chemistry_utils.load_species_table(
    general_data_path=general_species_data_path,
    species_names=species_names,
    energy_model_config=energy_model_config_gnoffo,
)

mesh = Mesh.from_1d_grid(jnp.array([0.0, dx]), periodic=True)
boundary_arrays = build_boundary_arrays_1d_periodic(mesh, species.n_species)

numerics_config = NumericsConfig(
    dt=dt,
    cfl=0.4,
    dt_mode="fixed",
    integrator_scheme="forward-euler",
    spatial_scheme="first_order",
    flux_scheme="hllc",
    clipping=ClippingConfig(),
)

equation_manager = EquationManager(
    species=species,
    reactions=None,
    numerics_config=numerics_config,
    boundary_arrays=boundary_arrays,
)

rho_init = p_init * species.molar_masses[0] / (constants.R_universal * T_tr_init)

# initial condition
U_init = compute_U_from_primitives(
    Y_s=jnp.array([[1.0]]),
    rho=jnp.array([rho_init]),
    u=jnp.array([0.0]),
    v=jnp.zeros(1),
    T_tr=jnp.array([T_tr_init]),
    T_V=jnp.array([T_V_init]),
    equation_manager=equation_manager,
)

# run simulation
U_field, t = run_scan(
    U_init=U_init,
    mesh=mesh,
    equation_manager=equation_manager,
    t_final=t_final,
    save_interval=save_interval,
)
print("Simulation completed.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Simulation completed.


In [15]:
from plotly import graph_objects as go
import pandas as pd

# JIT-compiled version of extract_primitives_from_U (needs to be executed once per kernel uptime)
def _extract_prim(U, em):
    prim = extract_primitives_from_U(U, em)
    return prim.Y_s, prim.rho, prim.T, prim.Tv, prim.p

extract_primitives_from_U_jitted = jax.jit(_extract_prim)

# extract primitives and plot
Y_s, rho, T, T_V, p = jax.vmap(
    extract_primitives_from_U_jitted,
    in_axes=(0, None),
)(U_field, equation_manager)

from plotly import graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=t,
        y=T_V[:, 0],
        mode="lines",
        name="T_V",
        line=dict(shape="spline", smoothing=1.0, width=4),
    )
)
fig.add_trace(
    go.Scatter(
        x=t, y=T[:, 0], mode="lines", name="T", line=dict(shape="spline", smoothing=1.0, width=4)
    )
)

fig.add_hline(y=7623.3, name="T_eq Casseau", line_dash="dash", line_color="black")

# Load and plot reference data from CSV
csv_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_1_cooling.csv"
df = pd.read_csv(
    csv_path, skiprows=1
)  # Skip the first header row, use X,Y,X,Y... as columns

# Read first row to get the dataset names
with open(csv_path, "r") as f:
    header_line = f.readline().strip()
dataset_names = [name for name in header_line.split(",") if name]

# Group datasets by prefix (e.g., "modified", "hy2foam_default", "monaco")
prefixes = []
for name in dataset_names:
    # Extract prefix by removing _t_v or _t_tr suffix
    if "_t_v" in name:
        prefix = name.replace("_t_v", "")
    elif "_t_tr" in name:
        prefix = name.replace("_t_tr", "")
    else:
        prefix = name
    if prefix not in prefixes:
        prefixes.append(prefix)

# Define colors for each prefix
colors = ["green", "red", "purple", "orange", "brown", "pink"]
prefix_colors = {prefix: colors[i % len(colors)] for i, prefix in enumerate(prefixes)}

# Plot each dataset (columns come in pairs: X, Y for each dataset)
for i, name in enumerate(dataset_names):
    x_col = i * 2  # X column index
    y_col = i * 2 + 1  # Y column index

    if x_col >= len(df.columns) or y_col >= len(df.columns):
        continue

    x_data = pd.to_numeric(df.iloc[:, x_col], errors="coerce").dropna().values
    y_data = pd.to_numeric(df.iloc[:, y_col], errors="coerce").dropna().values

    # Use minimum length in case of mismatched data
    min_len = min(len(x_data), len(y_data))
    x_data = x_data[:min_len]
    y_data = y_data[:min_len]

    # Find prefix for this dataset
    if "_t_v" in name:
        prefix = name.replace("_t_v", "")
    elif "_t_tr" in name:
        prefix = name.replace("_t_tr", "")
    else:
        prefix = name

    fig.add_trace(
        go.Scatter(
            x=x_data,
            y=y_data * 1000,  # Scale Y values (they appear to be in units of 1000 K)
            mode="markers+lines",
            line=dict(dash="dot", smoothing=1.0, shape="spline"),
            name=name,
            marker=dict(color=prefix_colors.get(prefix, "gray"), symbol="circle" if "_t_v" in name else "square" if "_t_tr" in name else "circle", size=12),
        )
    )

fig.update_xaxes(type="log", exponentformat="power", showexponent="all", showgrid=True)
fig.update_yaxes(range=[0, 10500], showgrid=True)
fig.update_layout(
    template="simple_white",
    # title="Energy Relaxation Correlation with Casseau for nonreacting N2",
    xaxis_title="Time (s)",
    yaxis_title="Temperature (K)",
    legend=dict(
        x=0.95,
        y=0.05,
        xanchor="right",
        yanchor="bottom",
        borderwidth=1,
        bordercolor="black",
    ),
    showlegend=True,
    width=800,
    height=800,
)
fig.update_layout(
    width=800,
    height=800,
    margin=dict(t=10, b=280, l=80, r=10),
    xaxis=dict(
        title_font=dict(size=26),
        tickfont=dict(size=22),
        tickangle=45,
    ),
    yaxis=dict(title_font=dict(size=26), tickfont=dict(size=22)),
    legend=dict(
        font=dict(size=22),
        x=0.5,
        y=-0.25,
        xanchor="center",
        yanchor="top",
        borderwidth=1,
        bordercolor="white",
        orientation="h",
    ),
)
fig.write_image(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/14.pdf"
)
fig.show()

## Case with excitation of electronic energy mode

In [6]:

%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp

import compressible.chemistry as chemistry
import compressible.chemistry_types as chemistry_types
import compressible.chemistry_utils as chemistry_utils
import compressible.constants as constants
import compressible.energy_models as energy_models
from compressible.boundary_conditions_utils import build_boundary_arrays_1d_periodic
from compressible.equation_manager import run_scan
from compressible.equation_manager_types import EquationManager
from compressible.mesh import Mesh
from compressible.numerics_types import ClippingConfig, NumericsConfig
from compressible.state import compute_U_from_primitives, extract_primitives_from_U

T_tr_init = 30000.0  # K
T_V_init = 1000.0  # K

p_init = 1.0 * constants.ATM_TO_PA

general_species_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_gnoffo.json"
)
gnoffo_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/"
    "air_5_gnoffo_equilibrium_enthalpy.json"
)
bird_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_bird_energy.json"
)
species_names = ["N2"]

dt = 1e-9  # s
t_final = 1e-5  # s
save_interval = 1  # every x steps
dx = 1e-4  # m  # TODO: show that this has no impact on the results in 0D

# ------------------------------ Simulation setup ------------------------------

energy_model_config_gnoffo = energy_models.EnergyModelConfig(
    model="gnoffo",
    include_electronic=True,
    data_path=gnoffo_equilibrium_enthalpy_data_path,
)

energy_model_config_bird = energy_models.EnergyModelConfig(
    model="bird",
    include_electronic=True,
    data_path=bird_equilibrium_enthalpy_data_path,
)

species = chemistry_utils.load_species_table(
    general_data_path=general_species_data_path,
    species_names=species_names,
    energy_model_config=energy_model_config_gnoffo,
)

mesh = Mesh.from_1d_grid(jnp.array([0.0, dx]), periodic=True)
boundary_arrays = build_boundary_arrays_1d_periodic(mesh, species.n_species)

numerics_config = NumericsConfig(
    dt=dt,
    cfl=0.4,
    dt_mode="fixed",
    integrator_scheme="rk2",
    spatial_scheme="first_order",
    flux_scheme="hllc",
    clipping=ClippingConfig(),
)

equation_manager = EquationManager(
    species=species,
    reactions=None,
    numerics_config=numerics_config,
    boundary_arrays=boundary_arrays,
)

rho_init = p_init * species.molar_masses[0] / (constants.R_universal * T_tr_init)

# initial condition
U_init = compute_U_from_primitives(
    Y_s=jnp.array([[1.0]]),
    rho=jnp.array([rho_init]),
    u=jnp.array([0.0]),
    v=jnp.zeros(1),
    T_tr=jnp.array([T_tr_init]),
    T_V=jnp.array([T_V_init]),
    equation_manager=equation_manager,
)

# run simulation
U_field, t = run_scan(
    U_init=U_init,
    mesh=mesh,
    equation_manager=equation_manager,
    t_final=t_final,
    save_interval=save_interval,
)
print("Simulation completed.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Simulation completed.


In [7]:
from plotly import graph_objects as go
import pandas as pd
import jax

# JIT-compiled version of extract_primitives_from_U (needs to be executed once per kernel uptime)
def _extract_prim(U, em):
    prim = extract_primitives_from_U(U, em)
    return prim.Y_s, prim.rho, prim.T, prim.Tv, prim.p

extract_primitives_from_U_jitted = jax.jit(_extract_prim)

# extract primitives and plot
Y_s, rho, T, T_V, p = jax.vmap(
    extract_primitives_from_U_jitted,
    in_axes=(0, None),
)(U_field, equation_manager)

from plotly import graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=t,
        y=T_V[:, 0],
        mode="lines",
        name="T_V",
        line=dict(shape="spline", smoothing=1.0, width=4),
    )
)
fig.add_trace(
    go.Scatter(
        x=t,
        y=T[:, 0],
        mode="lines",
        name="T",
        line=dict(shape="spline", smoothing=1.0, width=4),
    )
)

fig.add_hline(y=7623.3, name="T_eq Casseau", line_dash="dash", line_color="black")

# Load and plot reference data from CSV
csv_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_2.csv"
df = pd.read_csv(
    csv_path, skiprows=1
)  # Skip the first header row, use X,Y,X,Y... as columns

# Read first row to get the dataset names
with open(csv_path, "r") as f:
    header_line = f.readline().strip()
dataset_names = [name for name in header_line.split(",") if name]

# Group datasets by prefix (e.g., "modified", "hy2foam_default", "monaco")
prefixes = []
for name in dataset_names:
    # Extract prefix by removing _t_v or _t_tr suffix
    if "_t_v" in name:
        prefix = name.replace("_t_v", "")
    elif "_t_tr" in name:
        prefix = name.replace("_t_tr", "")
    else:
        prefix = name
    if prefix not in prefixes:
        prefixes.append(prefix)

# Define colors for each prefix
colors = ["green", "red", "purple", "orange", "brown", "pink"]
prefix_colors = {prefix: colors[i % len(colors)] for i, prefix in enumerate(prefixes)}

# Plot each dataset (columns come in pairs: X, Y for each dataset)
for i, name in enumerate(dataset_names):
    x_col = i * 2  # X column index
    y_col = i * 2 + 1  # Y column index

    if x_col >= len(df.columns) or y_col >= len(df.columns):
        continue

    x_data = pd.to_numeric(df.iloc[:, x_col], errors="coerce").dropna().values
    y_data = pd.to_numeric(df.iloc[:, y_col], errors="coerce").dropna().values

    # Use minimum length in case of mismatched data
    min_len = min(len(x_data), len(y_data))
    x_data = x_data[:min_len]
    y_data = y_data[:min_len]

    # Find prefix for this dataset
    if "_t_v" in name:
        prefix = name.replace("_t_v", "")
    elif "_t_tr" in name:
        prefix = name.replace("_t_tr", "")
    else:
        prefix = name

    fig.add_trace(
        go.Scatter(
            x=x_data,
            y=y_data * 1000,  # Scale Y values (they appear to be in units of 1000 K)
            mode="markers+lines",
            line=dict(dash="dot", smoothing=1.0, shape="spline"),
            name=name,
            marker=dict(color=prefix_colors.get(prefix, "gray"), symbol="circle" if "_t_v" in name else "square" if "_t_tr" in name else "circle", size=12),
        )
    )

fig.update_xaxes(type="log", exponentformat="power", showexponent="all", showgrid=True)
fig.update_yaxes(range=[0, 31000], showgrid=True)
fig.update_layout(
    template="simple_white",
    title="Energy Relaxation Correlation with Casseau for nonreacting N2 at high T",
    xaxis_title="Time (s)",
    yaxis_title="Temperature (K)",
    legend=dict(
        x=0.95,
        y=0.05,
        xanchor="right",
        yanchor="bottom",
        borderwidth=1,
        bordercolor="black",
    ),
    showlegend=True,
    width=800,
    height=800,
)
fig.update_layout(
    title=None,
    width=800,
    height=800,
    margin=dict(t=10, b=280, l=80, r=10),
    xaxis=dict(
        title_font=dict(size=26),
        tickfont=dict(size=22),
        tickangle=45,
    ),
    yaxis=dict(title_font=dict(size=26), tickfont=dict(size=22)),
    legend=dict(
        font=dict(size=22),
        x=0.5,
        y=-0.25,
        xanchor="center",
        yanchor="top",
        borderwidth=1,
        bordercolor="white",
        orientation="h",
    ),
)
fig.write_image(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/16.pdf"
)
fig.show()

## VT relaxation of non-reacting multi-species gas (T_tr > T_V)

In [22]:

%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp

import compressible.chemistry as chemistry
import compressible.chemistry_types as chemistry_types
import compressible.chemistry_utils as chemistry_utils
import compressible.constants as constants
import compressible.energy_models as energy_models
from compressible.boundary_conditions_utils import build_boundary_arrays_1d_periodic
from compressible.equation_manager import run_scan
from compressible.equation_manager_types import EquationManager
from compressible.mesh import Mesh
from compressible.numerics_types import ClippingConfig, NumericsConfig
from compressible.state import compute_U_from_primitives, extract_primitives_from_U

T_tr_init = 30000.0  # K
T_V_init = 1000.0  # K

p_init = 1.0 * constants.ATM_TO_PA

species_names = ["N2", "N"]
n_N2_init = 5.0e22  # 1/m3, number density of N2
n_N_init = 5.0e22  # 1/m3, number density of N

general_species_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_gnoffo.json"
)
gnoffo_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/"
    "air_5_gnoffo_equilibrium_enthalpy.json"
)
bird_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_bird_energy.json"
)

dt = 1e-9  # s
t_final = 1e-5  # s
save_interval = 1  # every x steps
dx = 1e-4  # m  # TODO: show that this has no impact on the results in 0D

# ------------------------------ Simulation setup ------------------------------

energy_model_config_gnoffo = energy_models.EnergyModelConfig(
    model="gnoffo",
    include_electronic=True,
    data_path=gnoffo_equilibrium_enthalpy_data_path,
)

energy_model_config_bird = energy_models.EnergyModelConfig(
    model="bird",
    include_electronic=False,
    data_path=bird_equilibrium_enthalpy_data_path,
)

species = chemistry_utils.load_species_table(
    general_data_path=general_species_data_path,
    species_names=species_names,
    energy_model_config=energy_model_config_bird,
)

Y_N2 = n_N2_init / (n_N2_init + n_N_init)
Y_N = n_N_init / (n_N2_init + n_N_init)

rho_N2 = (
    p_init
    * Y_N
    * species.molar_masses[species.names.index("N2")]
    / (constants.R_universal * T_tr_init)
)

rho_N = (
    p_init
    * Y_N
    * species.molar_masses[species.names.index("N")]
    / (constants.R_universal * T_tr_init)
)

mesh = Mesh.from_1d_grid(jnp.array([0.0, dx]), periodic=True)
boundary_arrays = build_boundary_arrays_1d_periodic(mesh, species.n_species)

numerics_config = NumericsConfig(
    dt=dt,
    cfl=0.4,
    dt_mode="fixed",
    integrator_scheme="forward-euler",
    spatial_scheme="first_order",
    flux_scheme="hllc",
    clipping=ClippingConfig(),
)

equation_manager = EquationManager(
    species=species,
    reactions=None,
    numerics_config=numerics_config,
    boundary_arrays=boundary_arrays,
)

# initial condition
U_init = compute_U_from_primitives(
    Y_s=jnp.array([[Y_N2, Y_N]]),
    rho=jnp.array([(rho_N2 + rho_N)]),
    u=jnp.array([0.0]),
    v=jnp.zeros(1),
    T_tr=jnp.array([T_tr_init]),
    T_V=jnp.array([T_V_init]),
    equation_manager=equation_manager,
)


# run simulation
U_field, t = run_scan(
    U_init=U_init,
    mesh=mesh,
    equation_manager=equation_manager,
    t_final=t_final,
    save_interval=save_interval,
)
print("Simulation completed.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Simulation completed.


In [23]:
from plotly import graph_objects as go
import pandas as pd

def _extract_prim(U, em):
    prim = extract_primitives_from_U(U, em)
    return prim.Y_s, prim.rho, prim.T, prim.Tv, prim.p

extract_primitives_from_U_jitted = jax.jit(_extract_prim)

_, _, T, T_V, _ = jax.vmap(extract_primitives_from_U_jitted, in_axes=(0, None))(
    U_field, equation_manager
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=t,
        y=T_V[:, 0],
        mode="lines",
        name="T_V",
        line=dict(shape="spline", smoothing=1.0, width=4),
    )
)
fig.add_trace(
    go.Scatter(
        x=t,
        y=T[:, 0],
        mode="lines",
        name="T",
        line=dict(dash="solid", shape="spline", smoothing=1.0, width=4),
    )
)

fig.add_hline(y=7623.3, name="T_eq Casseau", line_dash="dash", line_color="black")

# Load and plot hy2foam_default reference traces from CSV
csv_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_3.csv"
df = pd.read_csv(csv_path, skiprows=1)

with open(csv_path, "r") as f:
    header_line = f.readline().strip()
dataset_names = [name for name in header_line.split(",") if name]

for i, name in enumerate(dataset_names):
    if "hy2foam_default" not in name:
        continue

    x_col = i * 2
    y_col = i * 2 + 1

    if x_col >= len(df.columns) or y_col >= len(df.columns):
        continue

    x_data = pd.to_numeric(df.iloc[:, x_col], errors="coerce").dropna().values
    y_data = pd.to_numeric(df.iloc[:, y_col], errors="coerce").dropna().values

    min_len = min(len(x_data), len(y_data))

    fig.add_trace(
        go.Scatter(
            x=x_data[:min_len],
            y=y_data[:min_len] * 1000,
            mode="markers+lines",
            line=dict(color="green", dash="dot", smoothing=1.0, shape="spline"),
            name=name,
            marker=dict(color="green", symbol="circle" if "_t_v" in name else "square" if "_t_tr" in name else "circle", size=12),
        )
    )

fig.update_xaxes(type="log", exponentformat="power", showexponent="all", showgrid=True)
fig.update_yaxes(range=[0, 31000], showgrid=True)
fig.update_layout(
    template="simple_white",
    xaxis_title="Time (s)",
    yaxis_title="Temperature (K)",
    legend=dict(
        x=0.95,
        y=0.05,
        xanchor="right",
        yanchor="bottom",
        borderwidth=1,
        bordercolor="black",
    ),
    showlegend=True,
    width=800,
    height=600,
)
fig.show()
fig.update_layout(
    title=None,
    width=800,
    height=800,
    margin=dict(t=10, b=100, l=80, r=10),
    xaxis=dict(title_font=dict(size=22), tickfont=dict(size=18), tickangle=45),
    yaxis=dict(title_font=dict(size=22), tickfont=dict(size=18)),
    legend=dict(
        font=dict(size=18),
        x=0.5,
        y=-0.25,
        xanchor="center",
        yanchor="top",
        borderwidth=1,
        bordercolor="white",
        orientation="h",
    ),
)
fig.write_image(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/21.pdf"
)

## VT Relaxation of non-reacting multispecies gas (N2, O2) (T_tr < T_V)

Casseau correlates not only VT but also VV relaxation. Latter is not yet implemented.

In [19]:

%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp

import compressible.chemistry as chemistry
import compressible.chemistry_types as chemistry_types
import compressible.chemistry_utils as chemistry_utils
import compressible.constants as constants
import compressible.energy_models as energy_models
from compressible.boundary_conditions_utils import build_boundary_arrays_1d_periodic
from compressible.equation_manager import run_scan
from compressible.equation_manager_types import EquationManager
from compressible.mesh import Mesh
from compressible.numerics_types import ClippingConfig, NumericsConfig
from compressible.state import compute_U_from_primitives, extract_primitives_from_U

T_tr_init = 5000.0  # K
T_V_init = 30000.0  # K

p_init = 1.0 * constants.ATM_TO_PA

species_names = ["N2", "O2"]
n_N2_init = 5.0e22  # 1/m3, number density of N2
n_O2_init = 5.0e22  # 1/m3, number density of O2

general_species_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_gnoffo.json"
)
gnoffo_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/"
    "air_5_gnoffo_equilibrium_enthalpy.json"
)
bird_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_bird_energy.json"
)

dt = 1e-9  # s
t_final = 1e-5  # s
save_interval = 1  # every x steps
dx = 1e-4  # m  # TODO: show that this has no impact on the results in 0D

# ------------------------------ Simulation setup ------------------------------

energy_model_config_gnoffo = energy_models.EnergyModelConfig(
    model="gnoffo",
    include_electronic=True,
    data_path=gnoffo_equilibrium_enthalpy_data_path,
)

energy_model_config_bird = energy_models.EnergyModelConfig(
    model="bird",
    include_electronic=False,
    data_path=bird_equilibrium_enthalpy_data_path,
)

species = chemistry_utils.load_species_table(
    general_data_path=general_species_data_path,
    species_names=species_names,
    energy_model_config=energy_model_config_bird,
)

Y_N2 = n_N2_init / (n_N2_init + n_O2_init)
Y_O2 = n_O2_init / (n_N2_init + n_O2_init)

# rho_N2 = n_N2_init * species.molar_masses[species.names.index("N2")] / constants.N_A
# rho_O2 = n_N_init * species.molar_masses[species.names.index("O2")] / constants.N_A
rho_N2 = (
    p_init
    * Y_N2
    * species.molar_masses[species.names.index("N2")]
    / (constants.R_universal * T_tr_init)
)
rho_O2 = (
    p_init
    * Y_N2
    * species.molar_masses[species.names.index("O2")]
    / (constants.R_universal * T_tr_init)
)

mesh = Mesh.from_1d_grid(jnp.array([0.0, dx]), periodic=True)
boundary_arrays = build_boundary_arrays_1d_periodic(mesh, species.n_species)

numerics_config = NumericsConfig(
    dt=dt,
    cfl=0.4,
    dt_mode="fixed",
    integrator_scheme="forward-euler",
    spatial_scheme="first_order",
    flux_scheme="hllc",
    clipping=ClippingConfig(),
)

equation_manager = EquationManager(
    species=species,
    reactions=None,
    numerics_config=numerics_config,
    boundary_arrays=boundary_arrays,
)

# initial condition
U_init = compute_U_from_primitives(
    Y_s=jnp.array([[Y_N2, Y_O2]]),
    rho=jnp.array([(rho_N2 + rho_O2)]),
    u=jnp.array([0.0]),
    v=jnp.zeros(1),
    T_tr=jnp.array([T_tr_init]),
    T_V=jnp.array([T_V_init]),
    equation_manager=equation_manager,
)

# run simulation
U_field, t = run_scan(
    U_init=U_init,
    mesh=mesh,
    equation_manager=equation_manager,
    t_final=t_final,
    save_interval=save_interval,
)
print("Simulation completed.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Simulation completed.


In [20]:
from plotly import graph_objects as go
import pandas as pd

# JIT-compiled version of extract_primitives_from_U (needs to be executed once per kernel uptime)
def _extract_prim(U, em):
    prim = extract_primitives_from_U(U, em)
    return prim.Y_s, prim.rho, prim.T, prim.Tv, prim.p

extract_primitives_from_U_jitted = jax.jit(_extract_prim)

# extract primitives and plot
Y_s, rho, T, T_V, p = jax.vmap(
    extract_primitives_from_U_jitted,
    in_axes=(0, None),
)(U_field, equation_manager)

from plotly import graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=t,
        y=T_V[:, 0],
        mode="lines",
        name="T_V",
        line=dict(shape="spline", smoothing=1.0, width=4),
    )
)
fig.add_trace(
    go.Scatter(
        x=t,
        y=T[:, 0],
        mode="lines",
        name="T",
        line=dict(shape="spline", smoothing=1.0, width=4),
    )
)

fig.add_hline(y=7623.3, name="T_eq Casseau", line_dash="dash", line_color="black")

# Load and plot reference data from CSV
csv_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_4.csv"
df = pd.read_csv(
    csv_path, skiprows=1
)  # Skip the first header row, use X,Y,X,Y... as columns

# Read first row to get the dataset names
with open(csv_path, "r") as f:
    header_line = f.readline().strip()
dataset_names = [name for name in header_line.split(",") if name]

# Group datasets by prefix (e.g., "modified", "hy2foam_default", "monaco")
prefixes = []
for name in dataset_names:
    # Extract prefix by removing _t_v or _t_tr suffix
    if "_t_v" in name:
        prefix = name.replace("_t_v", "")
    elif "_t_tr" in name:
        prefix = name.replace("_t_tr", "")
    else:
        prefix = name
    if prefix not in prefixes:
        prefixes.append(prefix)

# Define colors for each prefix
colors = ["green", "red", "purple", "orange", "brown", "pink"]
prefix_colors = {prefix: colors[i % len(colors)] for i, prefix in enumerate(prefixes)}

# Plot each dataset (columns come in pairs: X, Y for each dataset)
for i, name in enumerate(dataset_names):
    x_col = i * 2  # X column index
    y_col = i * 2 + 1  # Y column index

    if x_col >= len(df.columns) or y_col >= len(df.columns):
        continue

    x_data = pd.to_numeric(df.iloc[:, x_col], errors="coerce").dropna().values
    y_data = pd.to_numeric(df.iloc[:, y_col], errors="coerce").dropna().values

    # Use minimum length in case of mismatched data
    min_len = min(len(x_data), len(y_data))
    x_data = x_data[:min_len]
    y_data = y_data[:min_len]

    # Find prefix for this dataset
    if "_t_v" in name:
        prefix = name.replace("_t_v", "")
    elif "_t_tr" in name:
        prefix = name.replace("_t_tr", "")
    else:
        prefix = name

    fig.add_trace(
        go.Scatter(
            x=x_data,
            y=y_data * 1000,  # Scale Y values (they appear to be in units of 1000 K)
            mode="markers+lines",
            line=dict(dash="dot", smoothing=1.0, shape="spline"),
            name=name,
            marker=dict(color=prefix_colors.get(prefix, "gray"), symbol="circle" if "_t_v" in name else "square" if "_t_tr" in name else "circle", size=12),
        )
    )

fig.update_xaxes(type="log", exponentformat="power", showexponent="all", showgrid=True)
fig.update_yaxes(range=[0, 31000], showgrid=True)
fig.update_layout(
    template="simple_white",
    title="Energy Relaxation Correlation with Casseau for nonreacting N2 and O2 at high T_V",
    xaxis_title="Time (s)",
    yaxis_title="Temperature (K)",
    legend=dict(
        x=0.95,
        y=0.05,
        xanchor="right",
        yanchor="bottom",
        borderwidth=1,
        bordercolor="black",
    ),
    showlegend=True,
    width=800,
    height=600,
)
fig.update_layout(
    title=None,
    width=800,
    height=800,
    margin=dict(t=10, b=100, l=80, r=10),
    xaxis=dict(title_font=dict(size=22), tickfont=dict(size=18), tickangle=45),
    yaxis=dict(title_font=dict(size=22), tickfont=dict(size=18)),
    legend=dict(
        font=dict(size=18),
        x=0.5,
        y=-0.25,
        xanchor="center",
        yanchor="top",
        borderwidth=1,
        bordercolor="white",
        orientation="h",
    ),
)
fig.write_image(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/22.pdf"
)
fig.show()

## Relaxation of a chemically reacting mixture

In [42]:

%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp

import compressible.chemistry as chemistry
import compressible.chemistry_types as chemistry_types
import compressible.chemistry_utils as chemistry_utils
import compressible.constants as constants
import compressible.energy_models as energy_models
from compressible.boundary_conditions_utils import build_boundary_arrays_1d_periodic
from compressible.equation_manager import run_scan
from compressible.equation_manager_types import EquationManager
from compressible.mesh import Mesh
from compressible.numerics_types import ClippingConfig, NumericsConfig
from compressible.state import compute_U_from_primitives, extract_primitives_from_U

T_tr_init = 30000.0  # K
T_V_init = 1000.0  # K

species_names = ["N2", "N"]
n_N2_init = 5.0e22  # 1/m3, number density of N2
n_N_init = 5.0e22  # 1/m3, number density of O2

p_init = 1.0 * constants.ATM_TO_PA

general_species_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/air_5_gnoffo.json"
)
gnoffo_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/"
    "air_5_gnoffo_equilibrium_enthalpy.json"
)
bird_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/air_5_bird_energy.json"
)
park_reaction_data_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/n2_reaction_set_park.json"
qk_reaction_data_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/n2_reaction_set_qk.json"

dt_fine = 1e-9
dt_coarse = 1e-7
t_switch = 1e-6
t_final = 1e-3  # s

n_fine = int(t_switch / dt_fine)  # 1000 steps at 1e-9
t_remaining = t_final - t_switch
n_coarse = int(t_remaining / dt_coarse)  # remaining steps at 1e-7

dt_array = jnp.concatenate(
    [
        jnp.full((n_fine,), dt_fine),
        jnp.full((n_coarse,), dt_coarse),
    ]
)
dt = dt_fine

save_interval = 1  # every x steps
dx = 1e-4  # m  # TODO: show that this has no impact on the results in 0D

# ------------------------------ Simulation setup ------------------------------

energy_model_config_gnoffo = energy_models.EnergyModelConfig(
    model="gnoffo",
    include_electronic=True,
    data_path=gnoffo_equilibrium_enthalpy_data_path,
)

energy_model_config_bird = energy_models.EnergyModelConfig(
    model="bird",
    include_electronic=False,
    data_path=bird_equilibrium_enthalpy_data_path,
)

species = chemistry_utils.load_species_table(
    general_data_path=general_species_data_path,
    species_names=species_names,
    energy_model_config=energy_model_config_bird,
)

# reaction_data_path = park_reaction_data_path
reaction_data_path = qk_reaction_data_path
# chemistry_model_config = chemistry_types.ChemistryModelConfig(model="park", park_vibrational_source="nonpreferential")
# chemistry_model_config = chemistry_types.ChemistryModelConfig(
#     model="park", park_vibrational_source="preferential_constant"
# )
chemistry_model_config = chemistry_types.ChemistryModelConfig(model="cvdv_qp")
reactions = chemistry_utils.load_reactions_from_json(
    json_path=reaction_data_path,
    species_table=species,
    chemistry_model_config=chemistry_model_config,
)

included_reactions, excluded_reactions = chemistry_utils.check_reaction_coverage(
    json_path=reaction_data_path, species_names=species.names
)
print("Included reactions:")
for rxn in included_reactions:
    print(f"  {rxn['equation']}")
print("\nExcluded reactions:")
for rxn in excluded_reactions:
    print(f"  {rxn['equation']} - Missing: {list(rxn['missing_species'])}")

Y_N2 = n_N2_init / (n_N2_init + n_N_init)
Y_N = n_N_init / (n_N2_init + n_N_init)

rho_N2 = n_N2_init * species.molar_masses[species.names.index("N2")] / constants.N_A
rho_N = n_N_init * species.molar_masses[species.names.index("N")] / constants.N_A

mesh = Mesh.from_1d_grid(jnp.array([0.0, dx]), periodic=True)
boundary_arrays = build_boundary_arrays_1d_periodic(mesh, species.n_species)

numerics_config = NumericsConfig(
    dt=dt,
    cfl=0.4,
    dt_mode="fixed",
    integrator_scheme="forward-euler",
    spatial_scheme="first_order",
    flux_scheme="hllc",
    clipping=ClippingConfig(),
)

equation_manager = EquationManager(
    species=species,
    reactions=reactions,
    numerics_config=numerics_config,
    boundary_arrays=boundary_arrays,
)

# initial condition
U_init = compute_U_from_primitives(
    Y_s=jnp.array([[Y_N2, Y_N]]),
    rho=jnp.array([(rho_N2 + rho_N)]),
    u=jnp.array([0.0]),
    v=jnp.zeros(1),
    T_tr=jnp.array([T_tr_init]),
    T_V=jnp.array([T_V_init]),
    equation_manager=equation_manager,
)

# run simulation
U_field, t = run_scan(
    U_init=U_init,
    mesh=mesh,
    equation_manager=equation_manager,
    t_final=t_final,
    save_interval=save_interval,
    dt_array=dt_array,
)
print("Simulation completed.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Included reactions:
  N2 + N2 -> 2N + N2

Excluded reactions:
Simulation completed.


In [29]:
U_field_parkpref = U_field
U_field_parknonpref = U_field
U_field_cvdv = U_field

In [38]:
U_field_parkpref = U_field

In [40]:
U_field_parknonpref = U_field

In [43]:
U_field_cvdv = U_field

In [ ]:
subsample_factor = 10
subsample_start_index = 10

runs = {
    "parkpref": U_field_parkpref,
    "parknonpref": U_field_parknonpref,
    "cvdv": U_field_cvdv,
}
run_colors = {"parkpref": "blue", "parknonpref": "red", "cvdv": "green"}


def subsample(U):
    return jnp.concatenate(
        [U[:subsample_start_index], U[subsample_start_index::subsample_factor]], axis=0
    )


t_plot = jnp.concatenate(
    [t[:subsample_start_index], t[subsample_start_index::subsample_factor]], axis=0
)

def _extract_prim(U, em):
    prim = extract_primitives_from_U(U, em)
    return prim.Y_s, prim.rho, prim.T, prim.Tv, prim.p

extract_primitives_from_U_jitted = jax.jit(_extract_prim)

# Precompute primitives for all runs
run_primitives = {}
for run_name, U_field in runs.items():
    Y_s, rho, T, T_V, p = jax.vmap(extract_primitives_from_U_jitted, in_axes=(0, None))(
        subsample(U_field), equation_manager
    )
    run_primitives[run_name] = (Y_s, rho, T, T_V, p)

from plotly import graph_objects as go
import pandas as pd


# ── helpers for reference CSV ─────────────────────────────────────────────────
def load_ref_csv(csv_path):
    df = pd.read_csv(csv_path, skiprows=1)
    with open(csv_path) as f:
        dataset_names = [n for n in f.readline().strip().split(",") if n]
    return df, dataset_names


def iter_ref_traces(df, dataset_names, suffix_map):
    """Yield (prefix, name, x_data, y_data) for each dataset."""
    for i, name in enumerate(dataset_names):
        x_col, y_col = i * 2, i * 2 + 1
        if y_col >= len(df.columns):
            continue
        x = pd.to_numeric(df.iloc[:, x_col], errors="coerce").dropna().values
        y = pd.to_numeric(df.iloc[:, y_col], errors="coerce").dropna().values
        n = min(len(x), len(y))
        prefix = name
        for suffix in suffix_map:
            if suffix in name:
                prefix = name.replace(suffix, "")
                break
        yield prefix, name, x[:n], y[:n]


def _temp_layout(fig, title, pdf_path):
    fig.update_xaxes(type="log", exponentformat="power", showexponent="all", showgrid=True)
    fig.update_yaxes(range=[0, 31000], showgrid=True)
    fig.update_layout(
        template="simple_white",
        title=title,
        xaxis_title="Time (s)",
        yaxis_title="Temperature (K)",
        legend=dict(x=0.95, y=0.95, xanchor="right", yanchor="top", borderwidth=1, bordercolor="black"),
        width=800,
        height=600,
    )
    fig.show()
    fig.update_layout(
        title=None,
        width=1200,
        height=600,
        margin=dict(t=10, b=100, l=80, r=10),
        xaxis=dict(title_font=dict(size=22), tickfont=dict(size=18), tickangle=45),
        yaxis=dict(title_font=dict(size=22), tickfont=dict(size=18)),
        legend=dict(
            font=dict(size=18),
            x=0.5,
            y=-0.25,
            xanchor="center",
            yanchor="top",
            borderwidth=1,
            bordercolor="white",
            orientation="h",
        ),
    )
    fig.write_image(pdf_path)


ref_colors = ["purple", "orange", "brown", "pink", "gray"]
df_temp, dataset_names_temp = load_ref_csv(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_5_temperature.csv"
)

# ── T (translational-rotational) plot ────────────────────────────────────────
fig = go.Figure()

for run_name, (_, _, T, _, _) in run_primitives.items():
    color = run_colors[run_name]
    fig.add_trace(
        go.Scatter(
            x=t_plot,
            y=T[:, 0],
            mode="lines",
            name=run_name,
            legendgroup=run_name,
            showlegend=True,
            line=dict(color=color, shape="spline", smoothing=1.0, width=4),
        )
    )

fig.add_hline(y=7623.3, name="T_eq Casseau", line_dash="dash", line_color="black")

seen_prefixes = {}
for prefix, name, x, y in iter_ref_traces(df_temp, dataset_names_temp, ["_t_v", "_t_tr"]):
    if "_t_tr" not in name:
        continue
    is_new = prefix not in seen_prefixes
    if is_new:
        seen_prefixes[prefix] = ref_colors[len(seen_prefixes) % len(ref_colors)]
    color = seen_prefixes[prefix]
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y * 1000,
            mode="markers+lines",
            name=prefix,
            legendgroup=f"ref_{prefix}",
            showlegend=is_new,
            line=dict(dash="dot", shape="spline", smoothing=1.0, color=color),
            marker=dict(color=color, symbol="square", size=12),
        )
    )

_temp_  layout(fig, "T (translational-rotational) — Casseau reacting N2/N",
             "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/31t.pdf")


# ── T_V (vibrational) plot ────────────────────────────────────────────────────
fig = go.Figure()

for run_name, (_, _, _, T_V, _) in run_primitives.items():
    color = run_colors[run_name]
    fig.add_trace(
        go.Scatter(
            x=t_plot,
            y=T_V[:, 0],
            mode="lines",
            name=run_name,
            legendgroup=run_name,
            showlegend=True,
            line=dict(color=color, shape="spline", smoothing=1.0, width=4),
        )
    )

seen_prefixes = {}
for prefix, name, x, y in iter_ref_traces(df_temp, dataset_names_temp, ["_t_v", "_t_tr"]):
    if "_t_v" not in name:
        continue
    is_new = prefix not in seen_prefixes
    if is_new:
        seen_prefixes[prefix] = ref_colors[len(seen_prefixes) % len(ref_colors)]
    color = seen_prefixes[prefix]
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y * 1000,
            mode="markers+lines",
            name=prefix,
            legendgroup=f"ref_{prefix}",
            showlegend=is_new,
            line=dict(dash="dot", shape="spline", smoothing=1.0, color=color),
            marker=dict(color=color, symbol="circle", size=12),
        )
    )

_temp_layout(fig, "T_V (vibrational) — Casseau reacting N2/N",
             "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/31v.pdf")


# ── Number density plot ───────────────────────────────────────────────────────
fig = go.Figure()

for run_name, (Y_s, rho, _, _, _) in run_primitives.items():
    color = run_colors[run_name]
    M = jnp.sum(Y_s * species.molar_masses[None, None, :], axis=2)
    n = Y_s[:, 0, :] * rho / M * constants.N_A
    n_tot = jnp.sum(n, axis=1)[0]
    n_N2 = n[:, species.names.index("N2")]
    n_N = n[:, species.names.index("N")]

    fig.add_trace(
        go.Scatter(
            x=t_plot,
            y=n_N2 / n_tot,
            mode="lines",
            name=run_name,
            legendgroup=run_name,
            showlegend=True,
            line=dict(color=color, shape="spline", smoothing=1.0, width=4),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=t_plot,
            y=n_N / n_tot,
            mode="lines",
            name=run_name,
            legendgroup=run_name,
            showlegend=False,
            line=dict(color=color, dash="dash", shape="spline", smoothing=1.0, width=4),
        )
    )

df, dataset_names = load_ref_csv(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_5_density.csv"
)
seen_prefixes = {}
for prefix, name, x, y in iter_ref_traces(df, dataset_names, ["_n2", "_n"]):
    is_new = prefix not in seen_prefixes
    if is_new:
        seen_prefixes[prefix] = ref_colors[len(seen_prefixes) % len(ref_colors)]
    color = seen_prefixes[prefix]
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="markers+lines",
            name=prefix,
            legendgroup=f"ref_{prefix}",
            showlegend=is_new,
            line=dict(dash="dot", shape="spline", smoothing=1.0, color=color),
            marker=dict(color=color),
        )
    )

fig.update_xaxes(type="log", exponentformat="power", showexponent="all", showgrid=True)
fig.update_layout(
    template="simple_white",
    title="Species number densities over time",
    xaxis_title="Time (s)",
    yaxis_title="normalized number density",
    legend=dict(
        x=0.05,
        y=0.95,
        xanchor="left",
        yanchor="top",
        borderwidth=1,
        bordercolor="black",
    ),
    width=800,
    height=600,
)
fig.show()
fig.update_layout(
    title=None,
    width=1200,
    height=600,
    margin=dict(t=10, b=100, l=80, r=10),
    xaxis=dict(title_font=dict(size=22), tickfont=dict(size=18), tickangle=45),
    yaxis=dict(title_font=dict(size=22), tickfont=dict(size=18)),
    legend=dict(
        font=dict(size=18),
        x=0.5,
        y=-0.25,
        xanchor="center",
        yanchor="top",
        borderwidth=1,
        bordercolor="white",
        orientation="h",
    ),
)
fig.write_image(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/31n.pdf"
)


In [82]:
subsample_factor = 10
subsample_start_index = 10

U_field_plot = jnp.concatenate(
    [
        U_field[:subsample_start_index],
        U_field[subsample_start_index::subsample_factor],
    ],
    axis=0,
)

t_plot = jnp.concatenate(
    [
        t[:subsample_start_index],
        t[subsample_start_index::subsample_factor],
    ],
    axis=0,
)

from plotly import graph_objects as go
import pandas as pd

def _extract_prim(U, em):
    prim = extract_primitives_from_U(U, em)
    return prim.Y_s, prim.rho, prim.T, prim.Tv, prim.p

extract_primitives_from_U_jitted = jax.jit(_extract_prim)

Y_s, rho, T, T_V, p = jax.vmap(
    extract_primitives_from_U_jitted,
    in_axes=(0, None),
)(U_field_plot, equation_manager)

csv_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_5_temperature.csv"
df_temp = pd.read_csv(csv_path, skiprows=1)
with open(csv_path) as f:
    dataset_names_temp = [n for n in f.readline().strip().split(",") if n]

prefixes = []
for name in dataset_names_temp:
    prefix = name.replace("_t_v", "").replace("_t_tr", "")
    if prefix not in prefixes:
        prefixes.append(prefix)

colors = ["green", "red", "purple", "orange", "brown", "pink"]
prefix_colors = {p: colors[i % len(colors)] for i, p in enumerate(prefixes)}


def _add_ref_traces(fig, suffix_filter, marker_symbol):
    for i, name in enumerate(dataset_names_temp):
        if suffix_filter not in name:
            continue
        x_col, y_col = i * 2, i * 2 + 1
        if y_col >= len(df_temp.columns):
            continue
        x = pd.to_numeric(df_temp.iloc[:, x_col], errors="coerce").dropna().values
        y = pd.to_numeric(df_temp.iloc[:, y_col], errors="coerce").dropna().values
        n = min(len(x), len(y))
        prefix = name.replace("_t_v", "").replace("_t_tr", "")
        fig.add_trace(
            go.Scatter(
                x=x[:n],
                y=y[:n] * 1000,
                mode="markers+lines",
                line=dict(dash="dot", smoothing=1.0, shape="spline"),
                name=name,
                marker=dict(color=prefix_colors.get(prefix, "gray"), symbol=marker_symbol, size=12),
            )
        )


def _temp_layout(fig, title, yrange):
    fig.update_xaxes(type="log", exponentformat="power", showexponent="all", showgrid=True)
    fig.update_yaxes(range=yrange, showgrid=True)
    fig.update_layout(
        template="simple_white",
        title=title,
        xaxis_title="Time (s)",
        yaxis_title="Temperature (K)",
        legend=dict(x=0.95, y=0.95, xanchor="right", yanchor="top", borderwidth=1, bordercolor="black"),
        showlegend=True,
        width=800,
        height=600,
    )
    fig.show()


# ── T (translational-rotational) ─────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_plot, y=T[:, 0], mode="lines+markers", name="T",
                          line=dict(shape="spline", smoothing=1.0, width=4)))
fig.add_hline(y=7623.3, name="T_eq Casseau", line_dash="dash", line_color="black")
_add_ref_traces(fig, "_t_tr", "square")
_temp_layout(fig, "T (translational-rotational) — Casseau reacting N2/N", [0, 31000])


# ── T_V (vibrational) ────────────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_plot, y=T_V[:, 0], mode="lines+markers", name="T_V",
                          line=dict(shape="spline", smoothing=1.0, width=4)))
_add_ref_traces(fig, "_t_v", "circle")
_temp_layout(fig, "T_V (vibrational) — Casseau reacting N2/N", [0, 31000])


# ── Number density ────────────────────────────────────────────────────────────
M = jnp.sum(Y_s * species.molar_masses[None, None, :], axis=2)
n = Y_s[:, 0, :] * rho / M * constants.N_A
n_tot = jnp.sum(n, axis=1)[0]
n_N2 = n[:, species.names.index("N2")]
n_N = n[:, species.names.index("N")]

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_plot, y=n_N2 / n_tot, mode="lines", name="N2",
                          legendgroup="N2", legendgrouptitle_text="N2",
                          line=dict(shape="spline", smoothing=1.0, width=4)))
fig.add_trace(go.Scatter(x=t_plot, y=n_N / n_tot, mode="lines", name="N",
                          legendgroup="N", legendgrouptitle_text="N",
                          line=dict(shape="spline", smoothing=1.0, width=4)))

csv_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_5_density.csv"
df = pd.read_csv(csv_path, skiprows=1)
with open(csv_path) as f:
    dataset_names = [n for n in f.readline().strip().split(",") if n]

prefixes_n = []
for name in dataset_names:
    prefix = name.replace("_n2", "").replace("_n", "")
    if prefix not in prefixes_n:
        prefixes_n.append(prefix)
prefix_colors_n = {p: colors[i % len(colors)] for i, p in enumerate(prefixes_n)}


def extract_species(name):
    name_l = name.lower()
    if "_n2" in name_l:
        return "N2"
    if "_n" in name_l:
        return "N"
    return None


for species_key in ["N2", "N"]:
    for i, name in enumerate(dataset_names):
        if extract_species(name) != species_key:
            continue
        x_col, y_col = i * 2, i * 2 + 1
        if y_col >= len(df.columns):
            continue
        x_data = pd.to_numeric(df.iloc[:, x_col], errors="coerce").dropna().values
        y_data = pd.to_numeric(df.iloc[:, y_col], errors="coerce").dropna().values
        n_pts = min(len(x_data), len(y_data))
        prefix = name.replace("_n2", "").replace("_n", "")
        fig.add_trace(
            go.Scatter(
                x=x_data[:n_pts],
                y=y_data[:n_pts],
                mode="markers+lines",
                line=dict(dash="dot", smoothing=1.0, shape="spline"),
                name=name,
                marker=dict(color=prefix_colors_n.get(prefix, "gray"), symbol="circle" if "_n2" in name else "square", size=12),
                legendgroup=species_key,
                showlegend=True,
            )
        )

fig.update_xaxes(type="log", exponentformat="power", showexponent="all", showgrid=True)
fig.update_layout(
    template="simple_white",
    title="Species number densities over time",
    xaxis_title="Time (s)",
    yaxis_title="normalized number density",
    legend=dict(x=0.05, y=0.95, xanchor="left", yanchor="top", borderwidth=1, bordercolor="black"),
    showlegend=True,
    width=800,
    height=600,
)
fig.show()


## Relaxation of chemically reacting mixture at thermal equilibrium

In [55]:

%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp

import compressible.chemistry as chemistry
import compressible.chemistry_types as chemistry_types
import compressible.chemistry_utils as chemistry_utils
import compressible.constants as constants
import compressible.energy_models as energy_models
from compressible.boundary_conditions_utils import build_boundary_arrays_1d_periodic
from compressible.equation_manager import run_scan
from compressible.equation_manager_types import EquationManager
from compressible.mesh import Mesh
from compressible.numerics_types import ClippingConfig, NumericsConfig
from compressible.state import compute_U_from_primitives, extract_primitives_from_U

T_tr_init = 30000.0  # K
T_V_init = 30000.0  # K

species_names = ["N2", "N"]
n_N2_init = 5.0e22  # 1/m3, number density of N2
n_N_init = 5.0e22  # 1/m3, number density of O2

general_species_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_gnoffo.json"
)
gnoffo_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/"
    "air_5_gnoffo_equilibrium_enthalpy.json"
)
bird_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_bird_energy.json"
)
park_reaction_data_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/n2_reaction_set_park.json"
qk_reaction_data_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/n2_reaction_set_qk.json"

dt_fine = 1e-9
dt_coarse = 1e-7
t_switch = 1e-6
t_final = 1e-4  # s

n_fine = int(t_switch / dt_fine)  # 1000 steps at 1e-9
t_remaining = t_final - t_switch
n_coarse = int(t_remaining / dt_coarse)  # remaining steps at 1e-7

dt_array = jnp.concatenate(
    [
        jnp.full((n_fine,), dt_fine),
        jnp.full((n_coarse,), dt_coarse),
    ]
)
dt = dt_fine

save_interval = 1  # every x steps
dx = 1e-4  # m  # TODO: show that this has no impact on the results in 0D

# ------------------------------ Simulation setup ------------------------------

energy_model_config_gnoffo = energy_models.EnergyModelConfig(
    model="gnoffo",
    include_electronic=True,
    data_path=gnoffo_equilibrium_enthalpy_data_path,
)

energy_model_config_bird = energy_models.EnergyModelConfig(
    model="bird",
    include_electronic=False,
    data_path=bird_equilibrium_enthalpy_data_path,
)

species = chemistry_utils.load_species_table(
    general_data_path=general_species_data_path,
    species_names=species_names,
    energy_model_config=energy_model_config_bird,
)

# reaction_data_path = park_reaction_data_path
reaction_data_path = qk_reaction_data_path
# chemistry_model_config = chemistry_types.ChemistryModelConfig(model="park", park_vibrational_source="nonpreferential")
# chemistry_model_config = chemistry_types.ChemistryModelConfig(
#     model="park", park_vibrational_source="preferential_constant"
# )
chemistry_model_config = chemistry_types.ChemistryModelConfig(model="cvdv_qp")
reactions = chemistry_utils.load_reactions_from_json(
    json_path=reaction_data_path,
    species_table=species,
    chemistry_model_config=chemistry_model_config,
)

included_reactions, excluded_reactions = chemistry_utils.check_reaction_coverage(
    json_path=reaction_data_path, species_names=species.names
)
print("Included reactions:")
for rxn in included_reactions:
    print(f"  {rxn['equation']}")
print("\nExcluded reactions:")
for rxn in excluded_reactions:
    print(f"  {rxn['equation']} - Missing: {list(rxn['missing_species'])}")

Y_N2 = n_N2_init / (n_N2_init + n_N_init)
Y_N = n_N_init / (n_N2_init + n_N_init)

rho_N2 = n_N2_init * species.molar_masses[species.names.index("N2")] / constants.N_A
rho_N = n_N_init * species.molar_masses[species.names.index("N")] / constants.N_A

mesh = Mesh.from_1d_grid(jnp.array([0.0, dx]), periodic=True)
boundary_arrays = build_boundary_arrays_1d_periodic(mesh, species.n_species)

numerics_config = NumericsConfig(
    dt=dt,
    cfl=0.4,
    dt_mode="fixed",
    integrator_scheme="forward-euler",
    spatial_scheme="first_order",
    flux_scheme="hllc",
    clipping=ClippingConfig(),
)

equation_manager = EquationManager(
    species=species,
    reactions=reactions,
    numerics_config=numerics_config,
    boundary_arrays=boundary_arrays,
)

# initial condition
U_init = compute_U_from_primitives(
    Y_s=jnp.array([[Y_N2, Y_N]]),
    rho=jnp.array([(rho_N2 + rho_N)]),
    u=jnp.array([0.0]),
    v=jnp.zeros(1),
    T_tr=jnp.array([T_tr_init]),
    T_V=jnp.array([T_V_init]),
    equation_manager=equation_manager,
)

# run simulation
U_field, t = run_scan(
    U_init=U_init,
    mesh=mesh,
    equation_manager=equation_manager,
    t_final=t_final,
    save_interval=save_interval,
    dt_array=dt_array,
)
print("Simulation completed.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Included reactions:
  N2 + N2 -> 2N + N2

Excluded reactions:
Simulation completed.


In [47]:
U_field_cvdv = U_field
U_field_pref = U_field
U_field_nonpref = U_field

In [56]:
U_field_cvdv = U_field

In [52]:
U_field_pref = U_field

In [54]:
U_field_nonpref = U_field

In [58]:
subsample_factor = 10
subsample_start_index = 10

runs = {
    "pref": U_field_pref,
    "nonpref": U_field_nonpref,
    "cvdv": U_field_cvdv,
}
run_colors = {"pref": "blue", "nonpref": "red", "cvdv": "green"}


def subsample(U):
    return jnp.concatenate(
        [U[:subsample_start_index], U[subsample_start_index::subsample_factor]], axis=0
    )


t_plot = jnp.concatenate(
    [t[:subsample_start_index], t[subsample_start_index::subsample_factor]], axis=0
)

def _extract_prim(U, em):
    prim = extract_primitives_from_U(U, em)
    return prim.Y_s, prim.rho, prim.T, prim.Tv, prim.p

extract_primitives_from_U_jitted = jax.jit(_extract_prim)

# Precompute primitives for all runs
run_primitives = {}
for run_name, U_field in runs.items():
    Y_s, rho, T, T_V, p = jax.vmap(extract_primitives_from_U_jitted, in_axes=(0, None))(
        subsample(U_field), equation_manager
    )
    run_primitives[run_name] = (Y_s, rho, T, T_V, p)

from plotly import graph_objects as go
import pandas as pd


def load_ref_csv(csv_path):
    df = pd.read_csv(csv_path, skiprows=1)
    with open(csv_path) as f:
        dataset_names = [n for n in f.readline().strip().split(",") if n]
    return df, dataset_names


def iter_ref_traces(df, dataset_names, suffixes):
    for i, name in enumerate(dataset_names):
        x_col, y_col = i * 2, i * 2 + 1
        if y_col >= len(df.columns):
            continue
        x = pd.to_numeric(df.iloc[:, x_col], errors="coerce").dropna().values
        y = pd.to_numeric(df.iloc[:, y_col], errors="coerce").dropna().values
        n = min(len(x), len(y))
        prefix = name
        for suffix in suffixes:
            if suffix in name:
                prefix = name.replace(suffix, "")
                break
        yield prefix, name, x[:n], y[:n]


def _temp_layout(fig, title, pdf_path, yrange):
    fig.update_xaxes(type="log", exponentformat="power", showexponent="all", showgrid=True)
    fig.update_yaxes(range=yrange, showgrid=True)
    fig.update_layout(
        template="simple_white",
        title=title,
        xaxis_title="Time (s)",
        yaxis_title="Temperature (K)",
        legend=dict(x=0.95, y=0.95, xanchor="right", yanchor="top", borderwidth=1, bordercolor="black"),
        width=800,
        height=600,
    )
    fig.show()
    fig.update_layout(
        title=None,
        width=1200,
        height=600,
        margin=dict(t=10, b=100, l=80, r=10),
        xaxis=dict(title_font=dict(size=22), tickfont=dict(size=18), tickangle=45),
        yaxis=dict(title_font=dict(size=22), tickfont=dict(size=18)),
        legend=dict(
            font=dict(size=18),
            x=0.5,
            y=-0.25,
            xanchor="center",
            yanchor="top",
            borderwidth=1,
            bordercolor="white",
            orientation="h",
        ),
    )
    fig.write_image(pdf_path)


ref_colors = ["purple", "orange", "brown", "pink", "gray"]
df_temp, dataset_names_temp = load_ref_csv(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_6_temperature.csv"
)

# ── T (translational-rotational) plot ────────────────────────────────────────
fig = go.Figure()

for run_name, (_, _, T, _, _) in run_primitives.items():
    color = run_colors[run_name]
    fig.add_trace(
        go.Scatter(
            x=t_plot,
            y=T[:, 0],
            mode="lines",
            name=run_name,
            legendgroup=run_name,
            showlegend=True,
            line=dict(color=color, shape="spline", smoothing=1.0, width=4),
        )
    )

seen_prefixes = {}
for prefix, name, x, y in iter_ref_traces(df_temp, dataset_names_temp, ["_t_v", "_t_tr"]):
    if "_t_tr" not in name:
        continue
    is_new = prefix not in seen_prefixes
    if is_new:
        seen_prefixes[prefix] = ref_colors[len(seen_prefixes) % len(ref_colors)]
    color = seen_prefixes[prefix]
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y * 1000,
            mode="markers+lines",
            name=prefix,
            legendgroup=f"ref_{prefix}",
            showlegend=is_new,
            line=dict(dash="dot", shape="spline", smoothing=1.0, color=color),
            marker=dict(color=color, symbol="square", size=12),
        )
    )

_temp_layout(fig, "T (translational-rotational) — Casseau nonreacting N2/O2 at high T_V",
             "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/32t.pdf",
             yrange=[8000, 40000])


# ── T_V (vibrational) plot ────────────────────────────────────────────────────
fig = go.Figure()

for run_name, (_, _, _, T_V, _) in run_primitives.items():
    color = run_colors[run_name]
    fig.add_trace(
        go.Scatter(
            x=t_plot,
            y=T_V[:, 0],
            mode="lines",
            name=run_name,
            legendgroup=run_name,
            showlegend=True,
            line=dict(color=color, shape="spline", smoothing=1.0, width=4),
        )
    )

seen_prefixes = {}
for prefix, name, x, y in iter_ref_traces(df_temp, dataset_names_temp, ["_t_v", "_t_tr"]):
    if "_t_v" not in name:
        continue
    is_new = prefix not in seen_prefixes
    if is_new:
        seen_prefixes[prefix] = ref_colors[len(seen_prefixes) % len(ref_colors)]
    color = seen_prefixes[prefix]
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y * 1000,
            mode="markers+lines",
            name=prefix,
            legendgroup=f"ref_{prefix}",
            showlegend=is_new,
            line=dict(dash="dot", shape="spline", smoothing=1.0, color=color),
            marker=dict(color=color, symbol="circle", size=12),
        )
    )

_temp_layout(fig, "T_V (vibrational) — Casseau nonreacting N2/O2 at high T_V",
             "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/32v.pdf",
             yrange=[8000, 40000])


# ── Number density plot ───────────────────────────────────────────────────────
fig = go.Figure()

for run_name, (Y_s, rho, _, _, _) in run_primitives.items():
    color = run_colors[run_name]
    M = jnp.sum(Y_s * species.molar_masses[None, None, :], axis=2)
    n = Y_s[:, 0, :] * rho / M * constants.N_A
    n_N2 = n[:, species.names.index("N2")]
    n_N = n[:, species.names.index("N")]

    fig.add_trace(
        go.Scatter(
            x=t_plot,
            y=n_N2 / (n_N2_init + n_N_init),
            mode="lines",
            name=run_name,
            legendgroup=run_name,
            showlegend=True,
            line=dict(color=color, shape="spline", smoothing=1.0, width=4),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=t_plot,
            y=n_N / (n_N2_init + n_N_init),
            mode="lines",
            name=run_name,
            legendgroup=run_name,
            showlegend=False,
            line=dict(color=color, dash="dash", shape="spline", smoothing=1.0, width=4),
        )
    )

df, dataset_names = load_ref_csv(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_6_density.csv"
)
seen_prefixes = {}
for prefix, name, x, y in iter_ref_traces(df, dataset_names, ["_n2", "_n"]):
    is_new = prefix not in seen_prefixes
    if is_new:
        seen_prefixes[prefix] = ref_colors[len(seen_prefixes) % len(ref_colors)]
    color = seen_prefixes[prefix]
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="markers+lines",
            name=prefix,
            legendgroup=f"ref_{prefix}",
            showlegend=is_new,
            line=dict(dash="dot", shape="spline", smoothing=1.0, color=color),
            marker=dict(color=color),
        )
    )

fig.add_trace(
    go.Scatter(
        x=[None], y=[None], mode="lines", name="N2",
        line=dict(color="black", dash="solid"), showlegend=True,
    )
)
fig.add_trace(
    go.Scatter(
        x=[None], y=[None], mode="lines", name="N",
        line=dict(color="black", dash="dash"), showlegend=True,
    )
)

fig.update_xaxes(type="log", exponentformat="power", showexponent="all", showgrid=True)
fig.update_layout(
    template="simple_white",
    title="Species number densities over time",
    xaxis_title="Time (s)",
    yaxis_title="normalized number density",
    legend=dict(
        x=0.05,
        y=0.95,
        xanchor="left",
        yanchor="top",
        borderwidth=1,
        bordercolor="black",
    ),
    width=800,
    height=600,
)
fig.show()
fig.update_layout(
    title=None,
    width=1200,
    height=600,
    margin=dict(t=10, b=100, l=80, r=10),
    xaxis=dict(title_font=dict(size=22), tickfont=dict(size=18), tickangle=45),
    yaxis=dict(title_font=dict(size=22), tickfont=dict(size=18)),
    legend=dict(
        font=dict(size=18),
        x=0.5,
        y=-0.25,
        xanchor="center",
        yanchor="top",
        borderwidth=1,
        bordercolor="white",
        orientation="h",
    ),
)
fig.write_image(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/32n.pdf"
)


In [51]:
subsample_factor = 10
subsample_start_index = 10

U_field_plot = jnp.concatenate(
    [
        U_field[:subsample_start_index],
        U_field[subsample_start_index::subsample_factor],
    ],
    axis=0,
)

t_plot = jnp.concatenate(
    [
        t[:subsample_start_index],
        t[subsample_start_index::subsample_factor],
    ],
    axis=0,
)


from plotly import graph_objects as go
import pandas as pd

# JIT-compiled version of extract_primitives_from_U (needs to be executed once per kernel uptime)
def _extract_prim(U, em):
    prim = extract_primitives_from_U(U, em)
    return prim.Y_s, prim.rho, prim.T, prim.Tv, prim.p

extract_primitives_from_U_jitted = jax.jit(_extract_prim)

# extract primitives and plot
Y_s, rho, T, T_V, p = jax.vmap(
    extract_primitives_from_U_jitted,
    in_axes=(0, None),
)(U_field_plot, equation_manager)

from plotly import graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=T_V[:, 0],
        mode="lines+markers",
        name="T_V",
        line=dict(shape="spline", smoothing=1.0, width=4),
    )
)
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=T[:, 0],
        mode="lines+markers",
        name="T",
        line=dict(shape="spline", smoothing=1.0, width=4),
    )
)


# Load and plot reference data from CSV
csv_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_6_temperature.csv"
df = pd.read_csv(
    csv_path, skiprows=1
)  # Skip the first header row, use X,Y,X,Y... as columns

# Read first row to get the dataset names
with open(csv_path, "r") as f:
    header_line = f.readline().strip()
dataset_names = [name for name in header_line.split(",") if name]

# Group datasets by prefix (e.g., "modified", "hy2foam_default", "monaco")
prefixes = []
for name in dataset_names:
    # Extract prefix by removing _t_v or _t_tr suffix
    if "_t_v" in name:
        prefix = name.replace("_t_v", "")
    elif "_t_tr" in name:
        prefix = name.replace("_t_tr", "")
    else:
        prefix = name
    if prefix not in prefixes:
        prefixes.append(prefix)

# Define colors for each prefix
colors = ["green", "red", "purple", "orange", "brown", "pink"]
prefix_colors = {prefix: colors[i % len(colors)] for i, prefix in enumerate(prefixes)}

# Plot each dataset (columns come in pairs: X, Y for each dataset)
for i, name in enumerate(dataset_names):
    x_col = i * 2  # X column index
    y_col = i * 2 + 1  # Y column index

    if x_col >= len(df.columns) or y_col >= len(df.columns):
        continue

    x_data = pd.to_numeric(df.iloc[:, x_col], errors="coerce").dropna().values
    y_data = pd.to_numeric(df.iloc[:, y_col], errors="coerce").dropna().values

    # Use minimum length in case of mismatched data
    min_len = min(len(x_data), len(y_data))
    x_data = x_data[:min_len]
    y_data = y_data[:min_len]

    # Find prefix for this dataset
    if "_t_v" in name:
        prefix = name.replace("_t_v", "")
    elif "_t_tr" in name:
        prefix = name.replace("_t_tr", "")
    else:
        prefix = name

    fig.add_trace(
        go.Scatter(
            x=x_data,
            y=y_data * 1000,  # Scale Y values (they appear to be in units of 1000 K)
            mode="markers+lines",
            line=dict(dash="dot", smoothing=1.0, shape="spline"),
            name=name,
            marker=dict(color=prefix_colors.get(prefix, "gray"), symbol="circle" if "_t_v" in name else "square" if "_t_tr" in name else "circle", size=12),
        )
    )


fig.update_yaxes(range=[0, 31000], showgrid=True)
fig.update_layout(
    template="simple_white",
    title="Energy Relaxation Correlation with Casseau for nonreacting N2 and O2 at high T_V",
    xaxis_title="Time (s)",
    yaxis_title="Temperature (K)",
    legend=dict(
        x=0.95,
        y=0.95,
        xanchor="right",
        yanchor="top",
        borderwidth=1,
        bordercolor="black",
    ),
    showlegend=True,
    width=800,
    height=600,
)
fig.update_xaxes(type="log", exponentformat="power", showexponent="all", showgrid=True)
fig.show()

# plot normalized species number densities
M = jnp.sum(Y_s * species.molar_masses[None, None, :], axis=2)
n = Y_s[:, 0, :] * rho / M * constants.N_A  # 1/m3


n_N2 = n[:, species.names.index("N2")]
n_N = n[:, species.names.index("N")]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=n_N2 / (n_N2_init + n_N_init),
        mode="lines+markers",
        name="N2 mole fraction",
        legendgroup="N2",
        legendgrouptitle_text="N2",
        line=dict(shape="spline", smoothing=1.0, width=4),
    )
)
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=n_N / (n_N2_init + n_N_init),
        mode="lines+markers",
        name="N mole fraction",
        legendgroup="N",
        legendgrouptitle_text="N",
        line=dict(shape="spline", smoothing=1.0, width=4),
    )
)

# Load and plot reference density data from CSV (normalized)
csv_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_6_density.csv"
df = pd.read_csv(csv_path, skiprows=1)

# Read first row to get the dataset names
with open(csv_path, "r") as f:
    header_line = f.readline().strip()
dataset_names = [name for name in header_line.split(",") if name]

# Group datasets by prefix (e.g., "hy2foam_default", "monaco")
prefixes = []
for name in dataset_names:
    if "_n2" in name:
        prefix = name.replace("_n2", "")
    elif "_n" in name:
        prefix = name.replace("_n", "")
    else:
        prefix = name
    if prefix not in prefixes:
        prefixes.append(prefix)

# Define colors for each prefix
colors = ["green", "red", "purple", "orange", "brown", "pink"]
prefix_colors = {prefix: colors[i % len(colors)] for i, prefix in enumerate(prefixes)}


def extract_species(name):
    name_l = name.lower()
    if "_n2" in name_l:
        return "N2"
    if "_n" in name_l:
        return "N"
    return None


# Plot each dataset (columns come in pairs: X, Y for each dataset)
for species_key in ["N2", "N"]:
    for i, name in enumerate(dataset_names):
        if extract_species(name) != species_key:
            continue
        x_col = i * 2
        y_col = i * 2 + 1

        if x_col >= len(df.columns) or y_col >= len(df.columns):
            continue

        x_data = pd.to_numeric(df.iloc[:, x_col], errors="coerce").dropna().values
        y_data = pd.to_numeric(df.iloc[:, y_col], errors="coerce").dropna().values

        min_len = min(len(x_data), len(y_data))
        x_data = x_data[:min_len]
        y_data = y_data[:min_len]

        # Prefix mapping
        if "_n2" in name:
            prefix = name.replace("_n2", "")
        elif "_n" in name:
            prefix = name.replace("_n", "")
        else:
            prefix = name

        fig.add_trace(
            go.Scatter(
                x=x_data,
                y=y_data,
                mode="markers+lines",
                line=dict(dash="dot", smoothing=1.0, shape="spline"),
                name=name,
                marker=dict(color=prefix_colors.get(prefix, "gray"), symbol="circle" if "_t_v" in name else "square" if "_t_tr" in name else "circle", size=12),
                legendgroup=species_key,
                showlegend=True,
            )
        )

fig.update_layout(
    template="simple_white",
    title="Species number densities over time",
    xaxis_title="Time (s)",
    yaxis_title="normalized number density",
    legend=dict(
        x=0.05,
        y=0.95,
        xanchor="left",
        yanchor="top",
        borderwidth=1,
        bordercolor="black",
    ),
    showlegend=True,
    width=800,
    height=600,
)
fig.update_xaxes(type="log", exponentformat="power", showexponent="all", showgrid=True)
fig.show()

## Chemically reacting air

In [69]:

%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp

import compressible.chemistry as chemistry
import compressible.chemistry_types as chemistry_types
import compressible.chemistry_utils as chemistry_utils
import compressible.constants as constants
import compressible.energy_models as energy_models
from compressible.boundary_conditions_utils import build_boundary_arrays_1d_periodic
from compressible.equation_manager import run_scan
from compressible.equation_manager_types import EquationManager
from compressible.mesh import Mesh
from compressible.numerics_types import ClippingConfig, NumericsConfig
from compressible.state import compute_U_from_primitives, extract_primitives_from_U

T_tr_init = 10000.0  # K
T_V_init = 10000.0  # K

species_names = ["N2", "N", "O2", "O", "NO"]

n_tot = 4.625e22  # 1/m3, total number density from n = p / (k_b * T) with p=0.063atm and T=10000K
n_N2_init = 0.78 * n_tot  # 1/m3, number density of N2
n_O2_init = 0.21 * n_tot  # 1/m3, number density of O2
n_N_init = 0.0  # 1/m3, number density of N
n_O_init = 0.0  # 1/m3, number density of O
n_NO_init = 0.0  # 1/m3, number density of NO

general_species_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_gnoffo.json"
)
gnoffo_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/"
    "air_5_gnoffo_equilibrium_enthalpy.json"
)
bird_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_bird_energy.json"
)
park_reaction_data_path = "/home/hhoechter/tum/jaxfluids_internship/data/park_reactions.json"
qk_reaction_data_path = "/home/hhoechter/tum/jaxfluids_internship/data/casseau_qk_reactions.json"

# for non-constant time stepping
dt_fine = 1e-9  # s
dt_coarse = 1e-9  # s
t_threshold = 1e-6
t_final = 1e-4  # s

n_fine = int(t_threshold / dt_fine)
n_coarse = int((t_final - t_threshold) / dt_coarse)

dt_array_fine = jnp.full((n_fine,), dt_fine)
dt_array_coarse = jnp.full((n_coarse,), dt_coarse)
dt_array = jnp.concatenate([dt_array_fine, dt_array_coarse])
dt = dt_fine

# for fixed time stepping
# dt = 1e-9  # s
# t_final = 1e-4  # s
# dt_array = None

save_interval = 1  # every x steps
dx = 1e-4  # m  # TODO: show that this has no impact on the results in 0D

# ------------------------------ Simulation setup ------------------------------

energy_model_config_gnoffo = energy_models.EnergyModelConfig(
    model="gnoffo",
    include_electronic=True,
    data_path=gnoffo_equilibrium_enthalpy_data_path,
)

energy_model_config_bird = energy_models.EnergyModelConfig(
    model="bird",
    include_electronic=False,
    data_path=bird_equilibrium_enthalpy_data_path,
)

species = chemistry_utils.load_species_table(
    general_data_path=general_species_data_path,
    species_names=species_names,
    energy_model_config=energy_model_config_bird,
)

# reaction_data_path = park_reaction_data_path
reaction_data_path = qk_reaction_data_path
# chemistry_model_config = chemistry_types.ChemistryModelConfig(model="park", park_vibrational_source="nonpreferential")
# chemistry_model_config = chemistry_types.ChemistryModelConfig(
#     model="park", park_vibrational_source="preferential_constant"
# )
chemistry_model_config = chemistry_types.ChemistryModelConfig(model="cvdv_qp")
reactions = chemistry_utils.load_reactions_from_json(
    json_path=reaction_data_path,
    species_table=species,
    chemistry_model_config=chemistry_model_config,
)

included_reactions, excluded_reactions = chemistry_utils.check_reaction_coverage(
    json_path=reaction_data_path, species_names=species.names
)
print("Included reactions:")
for rxn in included_reactions:
    print(f"  {rxn['equation']}")
print("\nExcluded reactions:")
for rxn in excluded_reactions:
    print(f"  {rxn['equation']} - Missing: {list(rxn['missing_species'])}")

Y_N2 = n_N2_init / (n_N2_init + n_N_init + n_O2_init + n_O_init + n_NO_init)
Y_N = n_N_init / (n_N2_init + n_N_init + n_O2_init + n_O_init + n_NO_init)
Y_O2 = n_O2_init / (n_N2_init + n_N_init + n_O2_init + n_O_init + n_NO_init)
Y_O = n_O_init / (n_N2_init + n_N_init + n_O2_init + n_O_init + n_NO_init)
Y_NO = n_NO_init / (n_N2_init + n_N_init + n_O2_init + n_O_init + n_NO_init)

rho_N2 = n_N2_init * species.molar_masses[species.names.index("N2")] / constants.N_A
rho_N = n_N_init * species.molar_masses[species.names.index("N")] / constants.N_A
rho_O2 = n_O2_init * species.molar_masses[species.names.index("O2")] / constants.N_A
rho_O = n_O_init * species.molar_masses[species.names.index("O")] / constants.N_A
rho_NO = n_NO_init * species.molar_masses[species.names.index("NO")] / constants.N_A

mesh = Mesh.from_1d_grid(jnp.array([0.0, dx]), periodic=True)
boundary_arrays = build_boundary_arrays_1d_periodic(mesh, species.n_species)

numerics_config = NumericsConfig(
    dt=dt,
    cfl=0.4,
    dt_mode="fixed",
    integrator_scheme="forward-euler",
    spatial_scheme="first_order",
    flux_scheme="hllc",
    clipping=ClippingConfig(),
)

equation_manager = EquationManager(
    species=species,
    reactions=reactions,
    numerics_config=numerics_config,
    boundary_arrays=boundary_arrays,
)

# initial condition
U_init = compute_U_from_primitives(
    Y_s=jnp.array([[Y_N2, Y_N, Y_O2, Y_O, Y_NO]]),
    rho=jnp.array([(rho_N2 + rho_N + rho_O2 + rho_O + rho_NO)]),
    u=jnp.array([0.0]),
    v=jnp.zeros(1),
    T_tr=jnp.array([T_tr_init]),
    T_V=jnp.array([T_V_init]),
    equation_manager=equation_manager,
)

# run simulation
U_field, t = run_scan(
    U_init=U_init,
    mesh=mesh,
    equation_manager=equation_manager,
    t_final=t_final,
    save_interval=save_interval,
    dt_array=dt_array,
)
print("Simulation completed.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Included reactions:
  O2 + N -> O + O + N
  O2 + NO -> O + O + NO
  O2 + N2 -> O + O + N2
  O2 + O2 -> O + O + O2
  O2 + O -> O + O + O
  N2 + O -> N + N + O
  N2 + O2 -> N + N + O2
  N2 + NO -> N + N + NO
  N2 + N2 -> N + N + N2
  N2 + N -> N + N + N
  NO + N2 -> N + O + N2
  NO + O2 -> N + O + O2
  NO + NO -> N + O + NO
  NO + O -> N + O + O
  NO + N -> N + O + N
  NO + O -> O2 + N
  N2 + O -> NO + N
  O2 + N -> NO + O
  NO + N -> N2 + O

Excluded reactions:
Simulation completed.


In [70]:
subsample_factor = 10
subsample_start_index = 10

U_field_plot = jnp.concatenate(
    [
        U_field[:subsample_start_index],
        U_field[subsample_start_index::subsample_factor],
    ],
    axis=0,
)

t_plot = jnp.concatenate(
    [
        t[:subsample_start_index],
        t[subsample_start_index::subsample_factor],
    ],
    axis=0,
)

from plotly import graph_objects as go
import pandas as pd

def _extract_prim(U, em):
    prim = extract_primitives_from_U(U, em)
    return prim.Y_s, prim.rho, prim.T, prim.Tv, prim.p

extract_primitives_from_U_jitted = jax.jit(_extract_prim)

Y_s, rho, T, T_V, p = jax.vmap(
    extract_primitives_from_U_jitted,
    in_axes=(0, None),
)(U_field_plot, equation_manager)

# ── Temperature plot ──────────────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_plot, y=T_V[:, 0], mode="lines", name="T_V",
                          line=dict(shape="spline", smoothing=1.0, width=4)))
fig.add_trace(go.Scatter(x=t_plot, y=T[:, 0], mode="lines", name="T",
                          line=dict(shape="spline", smoothing=1.0, width=4)))

csv_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_7_temperature.csv"
df_temp = pd.read_csv(csv_path, skiprows=1)
with open(csv_path) as f:
    dataset_names_temp = [n for n in f.readline().strip().split(",") if n]

prefixes = []
for name in dataset_names_temp:
    prefix = name.replace("_t_v", "").replace("_t_tr", "")
    if prefix not in prefixes:
        prefixes.append(prefix)

colors = ["green", "red", "purple", "orange", "brown", "pink"]
prefix_colors = {p: colors[i % len(colors)] for i, p in enumerate(prefixes)}

for i, name in enumerate(dataset_names_temp):
    x_col, y_col = i * 2, i * 2 + 1
    if y_col >= len(df_temp.columns):
        continue
    x = pd.to_numeric(df_temp.iloc[:, x_col], errors="coerce").dropna().values
    y = pd.to_numeric(df_temp.iloc[:, y_col], errors="coerce").dropna().values
    n = min(len(x), len(y))
    prefix = name.replace("_t_v", "").replace("_t_tr", "")
    fig.add_trace(
        go.Scatter(
            x=x[:n],
            y=y[:n] * 1000,
            mode="markers+lines",
            line=dict(dash="dot", smoothing=1.0, shape="spline"),
            name=name,
            marker=dict(color=prefix_colors.get(prefix, "gray"),
                        symbol="circle" if "_t_v" in name else "square" if "_t_tr" in name else "circle",
                        size=12),
        )
    )

fig.update_layout(
    template="simple_white",
    title="Energy Relaxation Correlation with Casseau for nonreacting N2 and O2 at high T_V",
    xaxis_title="Time (s)",
    yaxis_title="Temperature (K)",
    legend=dict(x=0.95, y=0.95, xanchor="right", yanchor="top", borderwidth=1, bordercolor="black"),
    showlegend=True,
    width=800,
    height=600,
)
# fig.show()
fig.update_xaxes(range=[-9, -3], type="log", exponentformat="power", showexponent="all", showgrid=True)
fig.update_yaxes(range=[5000, 10500], showgrid=True)
fig.show()
fig.update_layout(
    title=None,
    width=1200,
    height=800,
    margin=dict(t=10, b=100, l=80, r=10),
    xaxis=dict(title_font=dict(size=22), tickfont=dict(size=18), tickangle=45),
    yaxis=dict(title_font=dict(size=22), tickfont=dict(size=18)),
    legend=dict(font=dict(size=18), x=0.5, y=-0.25, xanchor="center", yanchor="top",
                borderwidth=1, bordercolor="white", orientation="h"),
)
fig.write_image(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/41t.pdf"
)


# ── Number density ────────────────────────────────────────────────────────────
M = jnp.sum(Y_s * species.molar_masses[None, None, :], axis=2)
n = Y_s[:, 0, :] * rho / M * constants.N_A

n_N2 = n[:, species.names.index("N2")]
n_N = n[:, species.names.index("N")]
n_O2 = n[:, species.names.index("O2")]
n_O = n[:, species.names.index("O")]
n_NO = n[:, species.names.index("NO")]

species_colors = {
    "N2": "#1f77b4",
    "N": "#ff7f0e",
    "O2": "#2ca02c",
    "O": "#d62728",
    "NO": "#9467bd",
}

denom = n_N2_init + n_N_init + n_O2_init + n_O_init + n_NO_init

fig = go.Figure()
for sp, arr in [("N2", n_N2), ("N", n_N), ("O2", n_O2), ("O", n_O), ("NO", n_NO)]:
    fig.add_trace(go.Scatter(
        x=t_plot, y=arr / denom, mode="lines", name=f"{sp} mole fraction",
        legendgroup=sp, legendgrouptitle_text=sp,
        line=dict(shape="spline", smoothing=1.0, color=species_colors[sp], width=4),
    ))

csv_path = "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/casseau_figure_3_7_density.csv"
df = pd.read_csv(csv_path, skiprows=1)
with open(csv_path) as f:
    dataset_names = [n for n in f.readline().strip().split(",") if n]


def extract_species(name):
    name_l = name.lower()
    if "_n2" in name_l: return "N2"
    if "_o2" in name_l: return "O2"
    if "_no" in name_l: return "NO"
    if "_n" in name_l: return "N"
    if "_o" in name_l: return "O"
    return None


for species_key in ["N2", "N", "O2", "O", "NO"]:
    for i, name in enumerate(dataset_names):
        if extract_species(name) != species_key:
            continue
        x_col, y_col = i * 2, i * 2 + 1
        if y_col >= len(df.columns):
            continue
        x_data = pd.to_numeric(df.iloc[:, x_col], errors="coerce").dropna().values
        y_data = pd.to_numeric(df.iloc[:, y_col], errors="coerce").dropna().values
        n_pts = min(len(x_data), len(y_data))
        color = species_colors.get(species_key, "gray")
        if name.startswith("dsmcfoam"):
            mode, line, marker = "markers", dict(color=color), dict(symbol="triangle-up", size=10, color=color)
        elif name.startswith("hyfoam"):
            mode, line, marker = "markers+lines", dict(dash="dot", shape="spline", smoothing=1.0, color=color), dict(symbol="circle", size=8, color=color)
        else:
            mode, line, marker = "markers+lines", dict(dash="dash", shape="spline", smoothing=1.0, color=color), dict(symbol="square", size=8, color=color)
        fig.add_trace(go.Scatter(
            x=x_data[:n_pts], y=y_data[:n_pts], mode=mode, name=name,
            line=line, marker=marker, legendgroup=species_key, showlegend=True,
        ))

fig.update_layout(
    template="simple_white",
    title="Species number densities over time",
    xaxis_title="Time (s)",
    yaxis_title="normalized number density",
    legend=dict(x=0.5, y=-0.15, xanchor="center", yanchor="top", orientation="h"),
    showlegend=True,
    width=800,
    height=800,
)
fig.update_xaxes(range=[0, 1e-5])
# fig.show()
fig.update_xaxes(range=[-9, -3], type="log", exponentformat="power", showexponent="all", showgrid=True)
fig.update_yaxes(range=[-5, 0], type="log", exponentformat="power", showexponent="all", showgrid=True)
fig.show()
fig.update_layout(
    title=None,
    width=1200,
    height=800,
    margin=dict(t=10, b=100, l=80, r=10),
    xaxis=dict(title_font=dict(size=22), tickfont=dict(size=18), tickangle=45),
    yaxis=dict(title_font=dict(size=22), tickfont=dict(size=18)),
    legend=dict(font=dict(size=18), x=0.5, y=-0.25, xanchor="center", yanchor="top",
                borderwidth=1, bordercolor="white", orientation="h"),
)
fig.write_image(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/41n.pdf"
)


In [3]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

files = {
    "casseau_qk": "/home/hhoechter/tum/jaxfluids_internship/data/casseau_qk_reactions.json",
    "park": "/home/hhoechter/tum/jaxfluids_internship/data/park_reactions.json",
    "park1990": "/home/hhoechter/tum/jaxfluids_internship/data/park_1990_reactions.json",
    "park1993": "/home/hhoechter/tum/jaxfluids_internship/data/park_1993_reactions.json",
    "scanlon": "/home/hhoechter/tum/jaxfluids_internship/data/scanlon_table2_reactions.json",
}


def load_rates(path):
    data = json.loads(Path(path).read_text())
    rates = {}
    for rxn in data.get("reactions", []):
        eq = rxn.get("equation", "")
        rates[eq] = {
            "C_f": rxn.get("C_f"),
            "n_f": rxn.get("n_f"),
            "E_f_over_k": rxn.get("E_f_over_k"),
        }
    return rates


rates_by_source = {name: load_rates(path) for name, path in files.items()}

# Union of reactions (preserve order from first source)
order = []
seen = set()
for src in files.keys():
    for eq in rates_by_source[src]:
        if eq not in seen:
            order.append(eq)
            seen.add(eq)


def build_df(param):
    rows = []
    for eq in order:
        row = {
            src: rates_by_source[src].get(eq, {}).get(param, np.nan)
            for src in files.keys()
        }
        rows.append(row)
    return pd.DataFrame(rows, index=order)


df_Cf = build_df("C_f")
df_nf = build_df("n_f")
df_Ek = build_df("E_f_over_k")


def row_colors(df, log_scale=False):
    colors = []
    for _, row in df.iterrows():
        vals = row.values.astype(float)
        mask = np.isfinite(vals)
        if log_scale:
            vals = np.where(vals > 0, np.log10(vals), np.nan)
            mask = np.isfinite(vals)
        if mask.sum() == 0:
            colors.append(["#f0f0f0"] * len(row))
            continue
        v = vals[mask]
        vmin, vmax = v.min(), v.max()
        if np.isclose(vmin, vmax):
            norm = np.full_like(vals, 0.5, dtype=float)
        else:
            norm = (vals - vmin) / (vmax - vmin)
        row_colors = []
        for i, ok in enumerate(mask):
            if not ok:
                row_colors.append("#f0f0f0")
            else:
                # viridis-ish from blue->yellow
                row_colors.append(
                    f"rgb({int(68+187*norm[i])},{int(1+180*norm[i])},{int(84+20*(1-norm[i]))})"
                )
        colors.append(row_colors)
    return colors


def format_vals(df):
    return df.applymap(lambda x: "" if not np.isfinite(x) else f"{x:.4g}")


def luminance(rgb):
    # rgb is tuple of ints (0-255)
    r, g, b = rgb
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def row_colors_and_font(df, log_scale=False):
    colors = []
    fonts = []
    for _, row in df.iterrows():
        vals = row.values.astype(float)
        mask = np.isfinite(vals)
        if log_scale:
            vals = np.where(vals > 0, np.log10(vals), np.nan)
            mask = np.isfinite(vals)
        if mask.sum() == 0:
            colors.append(["#f0f0f0"] * len(row))
            fonts.append(["black"] * len(row))
            continue
        v = vals[mask]
        vmin, vmax = v.min(), v.max()
        if np.isclose(vmin, vmax):
            norm = np.full_like(vals, 0.5, dtype=float)
        else:
            norm = (vals - vmin) / (vmax - vmin)
        row_colors = []
        row_fonts = []
        for i, ok in enumerate(mask):
            if not ok:
                row_colors.append("#f0f0f0")
                row_fonts.append("black")
            else:
                # viridis-ish (same as before)
                r = int(68 + 187 * norm[i])
                g = int(1 + 180 * norm[i])
                b = int(84 + 20 * (1 - norm[i]))
                row_colors.append(f"rgb({r},{g},{b})")
                row_fonts.append("white" if luminance((r, g, b)) < 140 else "black")
        colors.append(row_colors)
        fonts.append(row_fonts)
    return colors, fonts


def make_table(df, title, log_scale):
    cell_text = [df.index.tolist()] + [
        format_vals(df)[col].tolist() for col in df.columns
    ]
    cell_colors = [["#ffffff"] * len(df.index)]
    cell_fonts = [["black"] * len(df.index)]

    rowwise_colors, rowwise_fonts = row_colors_and_font(df, log_scale=log_scale)
    for col_i in range(len(df.columns)):
        cell_colors.append(
            [rowwise_colors[row_i][col_i] for row_i in range(len(df.index))]
        )
        cell_fonts.append(
            [rowwise_fonts[row_i][col_i] for row_i in range(len(df.index))]
        )

    return go.Table(
        header=dict(
            values=["Reaction"] + list(df.columns),
            fill_color="#dddddd",
            align="left",
            font=dict(size=12, color="black"),
        ),
        cells=dict(
            values=cell_text,
            fill_color=cell_colors,
            font=dict(size=10, color=cell_fonts),
            align="left",
        ),
        name=title,
    )


fig = make_subplots(
    rows=1,
    cols=3,
    specs=[[{"type": "table"}, {"type": "table"}, {"type": "table"}]],
    subplot_titles=["C_f", "n_f", "E_f_over_k"],
)

fig.add_trace(make_table(df_Cf, "C_f", log_scale=True), row=1, col=1)
fig.add_trace(make_table(df_nf, "n_f", log_scale=False), row=1, col=2)
fig.add_trace(make_table(df_Ek, "E_f_over_k", log_scale=True), row=1, col=3)

fig.update_layout(
    height=1200,
    width=2000,
    title_text="Arrhenius Coefficients Comparison (row-wise normalized colors)",
)

fig.show()

/tmp/ipykernel_70798/193781230.py:89: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



In [16]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go

files = {
    "casseau_qk": "/home/hhoechter/tum/jaxfluids_internship/data/casseau_qk_reactions.json",
    "park": "/home/hhoechter/tum/jaxfluids_internship/data/park_reactions.json",
    "park1990": "/home/hhoechter/tum/jaxfluids_internship/data/park_1990_reactions.json",
    "park1993": "/home/hhoechter/tum/jaxfluids_internship/data/park_1993_reactions.json",
    "scanlon": "/home/hhoechter/tum/jaxfluids_internship/data/scanlon_table2_reactions.json",
}


def load_rates(path):
    data = json.loads(Path(path).read_text())
    rates = {}
    for rxn in data.get("reactions", []):
        eq = rxn.get("equation", "")
        rates[eq] = {
            "C_f": rxn.get("C_f"),
            "n_f": rxn.get("n_f"),
            "E_f_over_k": rxn.get("E_f_over_k"),
        }
    return rates


rates_by_source = {name: load_rates(path) for name, path in files.items()}

order = []
seen = set()
for src in files.keys():
    for eq in rates_by_source[src]:
        if eq not in seen:
            order.append(eq)
            seen.add(eq)


def build_df(param):
    rows = []
    for eq in order:
        row = {
            src: rates_by_source[src].get(eq, {}).get(param, np.nan)
            for src in files.keys()
        }
        rows.append(row)
    return pd.DataFrame(rows, index=order)


df_Cf = build_df("C_f")
df_nf = build_df("n_f")
df_Ek = build_df("E_f_over_k")


def luminance(r, g, b):
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def viridis_colors(df, log_scale=False):
    """Row-wise viridis-approximation coloring. Returns (fill_colors, font_colors) per column."""
    n_rows, n_cols = df.shape
    fill = [["#f0f0f0"] * n_rows for _ in range(n_cols)]
    font = [["black"] * n_rows for _ in range(n_cols)]

    for row_i, (_, row) in enumerate(df.iterrows()):
        vals = row.values.astype(float)
        mask = np.isfinite(vals)
        if log_scale:
            vals = np.where(vals > 0, np.log10(vals), np.nan)
            mask = np.isfinite(vals)
        if mask.sum() == 0:
            continue
        v = vals[mask]
        vmin, vmax = v.min(), v.max()
        norm = (
            np.full_like(vals, 0.5, dtype=float)
            if np.isclose(vmin, vmax)
            else (vals - vmin) / (vmax - vmin)
        )
        for col_i, ok in enumerate(mask):
            if not ok:
                continue
            r = int(68 + 187 * norm[col_i])
            g = int(1 + 180 * norm[col_i])
            b = int(84 + 20 * (1 - norm[col_i]))
            fill[col_i][row_i] = f"rgb({r},{g},{b})"
            font[col_i][row_i] = "white" if luminance(r, g, b) < 140 else "black"

    return fill, font


def fmt(df):
    return df.applymap(lambda x: "" if not np.isfinite(x) else f"{x:.4g}")


sources = list(files.keys())
n_src = len(sources)

# ── build column data ─────────────────────────────────────────────────────────
# columns: reaction | C_f x5 | n_f x5 | E_f_over_k (park1993 only)
reaction_col = order

cf_fmt = fmt(df_Cf)
nf_fmt = fmt(df_nf)
ek_1993 = [
    ""
    if not np.isfinite(df_Ek.loc[eq, "park1993"])
    else f"{df_Ek.loc[eq, 'park1993']:.4g}"
    for eq in order
]

cf_fill, cf_font = viridis_colors(df_Cf, log_scale=True)
nf_fill, nf_font = viridis_colors(df_nf, log_scale=False)

plain_fill = ["#ffffff"] * len(order)
plain_font = ["black"] * len(order)

# Plotly table: values[i] = column i
values = [reaction_col]
fill_colors = [plain_fill]
font_colors = [plain_font]
headers = ["Reaction"]

for i, src in enumerate(sources):
    values.append(cf_fmt[src].tolist())
    fill_colors.append(cf_fill[i])
    font_colors.append(cf_font[i])
    headers.append(f"C_f<br>{src}")

for i, src in enumerate(sources):
    values.append(nf_fmt[src].tolist())
    fill_colors.append(nf_fill[i])
    font_colors.append(nf_font[i])
    headers.append(f"n_f<br>{src}")

values.append(ek_1993)
fill_colors.append(plain_fill)
font_colors.append(plain_font)
headers.append("E/k [K]<br>park1993")

# ── column widths: reaction column gets 3x the space ─────────────────────────
col_widths = [2] + [1] * n_src + [1] * n_src + [1]

# ── header colors: grey | blue group (C_f) | orange group (n_f) | green (E/k)
header_fill = ["#dddddd"] + ["#b8cfe8"] * n_src + ["#e8d0b0"] * n_src + ["#c8e0c8"]

fig = go.Figure(
    go.Table(
        header=dict(
            values=headers,
            fill_color=header_fill,
            align="center",
            font=dict(size=10, color="black"),
        ),
        cells=dict(
            values=values,
            fill_color=fill_colors,
            font=dict(size=9, color=font_colors),
            align="left",
            height=20,
        ),
        columnwidth=col_widths,
    )
)

fig.update_layout(
    height=600,
    width=900,
    margin=dict(t=30, b=10, l=10, r=10),
)
fig.show()
fig.write_image(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/heatbath_0d_casseau/arrhenius_comparison.pdf"
)

/tmp/ipykernel_70798/1390965974.py:91: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



In [7]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

files = {
    "casseau_qk": "/home/hhoechter/tum/jaxfluids_internship/data/casseau_qk_reactions.json",
    # "park": "/home/hhoechter/tum/jaxfluids_internship/data/park_reactions.json",
    "park_1993": "/home/hhoechter/tum/jaxfluids_internship/data/park_1993_reactions.json",
    "park_1990": "/home/hhoechter/tum/jaxfluids_internship/data/park_1990_reactions.json",
    "scanlon": "/home/hhoechter/tum/jaxfluids_internship/data/scanlon_table2_reactions.json",
}


def load_rates(path):
    data = json.loads(Path(path).read_text())
    rates = {}
    for rxn in data.get("reactions", []):
        eq = rxn.get("equation", "")
        rates[eq] = {
            "C_f": rxn.get("C_f"),
            "n_f": rxn.get("n_f"),
            "E_f_over_k": rxn.get("E_f_over_k"),
        }
    return rates


rates_by_source = {name: load_rates(path) for name, path in files.items()}

# Union of reactions (preserve order from first source)
order = []
seen = set()
for src in files.keys():
    for eq in rates_by_source[src]:
        if eq not in seen:
            order.append(eq)
            seen.add(eq)


def build_df(param):
    rows = []
    for eq in order:
        row = {
            src: rates_by_source[src].get(eq, {}).get(param, np.nan)
            for src in files.keys()
        }
        rows.append(row)
    return pd.DataFrame(rows, index=order)


df_Cf = build_df("C_f")
df_nf = build_df("n_f")
df_Ek = build_df("E_f_over_k")


def luminance(rgb):
    r, g, b = rgb
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def global_colors_and_font(df, log_scale=False):
    vals = df.values.astype(float)
    if log_scale:
        vals = np.where(vals > 0, np.log10(vals), np.nan)

    mask = np.isfinite(vals)
    if mask.sum() == 0:
        return [["#f0f0f0"] * df.shape[1] for _ in range(df.shape[0])], [
            ["black"] * df.shape[1] for _ in range(df.shape[0])
        ]

    vmin, vmax = np.nanmin(vals), np.nanmax(vals)
    if np.isclose(vmin, vmax):
        norm = np.full_like(vals, 0.5, dtype=float)
    else:
        norm = (vals - vmin) / (vmax - vmin)

    colors = []
    fonts = []
    for i in range(df.shape[0]):
        row_colors = []
        row_fonts = []
        for j in range(df.shape[1]):
            if not np.isfinite(vals[i, j]):
                row_colors.append("#f0f0f0")
                row_fonts.append("black")
                continue
            r = int(68 + 187 * norm[i, j])
            g = int(1 + 180 * norm[i, j])
            b = int(84 + 20 * (1 - norm[i, j]))
            row_colors.append(f"rgb({r},{g},{b})")
            row_fonts.append("white" if luminance((r, g, b)) < 140 else "black")
        colors.append(row_colors)
        fonts.append(row_fonts)
    return colors, fonts


def format_vals(df):
    return df.applymap(lambda x: "" if not np.isfinite(x) else f"{x:.4g}")


def make_table(df, title, log_scale):
    cell_text = [df.index.tolist()] + [
        format_vals(df)[col].tolist() for col in df.columns
    ]
    cell_colors = [["#ffffff"] * len(df.index)]
    cell_fonts = [["black"] * len(df.index)]

    colors, fonts = global_colors_and_font(df, log_scale=log_scale)
    for col_i in range(len(df.columns)):
        cell_colors.append([colors[row_i][col_i] for row_i in range(len(df.index))])
        cell_fonts.append([fonts[row_i][col_i] for row_i in range(len(df.index))])

    return go.Table(
        header=dict(
            values=["Reaction"] + list(df.columns),
            fill_color="#dddddd",
            align="left",
            font=dict(size=12, color="black"),
        ),
        cells=dict(
            values=cell_text,
            fill_color=cell_colors,
            font=dict(size=10, color=cell_fonts),
            align="left",
        ),
        name=title,
    )


fig = make_subplots(
    rows=1,
    cols=3,
    specs=[[{"type": "table"}, {"type": "table"}, {"type": "table"}]],
    subplot_titles=[
        "C_f (global color)",
        "n_f (global color)",
        "E_f_over_k (global color)",
    ],
)

fig.add_trace(make_table(df_Cf, "C_f", log_scale=True), row=1, col=1)
fig.add_trace(make_table(df_nf, "n_f", log_scale=False), row=1, col=2)
fig.add_trace(make_table(df_Ek, "E_f_over_k", log_scale=True), row=1, col=3)

fig.update_layout(
    height=1200,
    width=1400,
    title_text="Arrhenius Coefficients (global normalization across reactions)",
)
fig.show()

/tmp/ipykernel_70798/1164619143.py:99: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

